# VQA competition — self-contained submission notebook

Runs top-to-bottom with no external repo: cell 1 writes the `src/` package and
`configs/` to disk (embedded, base64), then normal `python -m src.train` / `src.predict`
calls train and produce `submission.npy` + `model.pt`. Generated by
`tools/build_submission_notebook.py` — edit `src/`, regenerate, do not hand-edit.

**Prereq:** `data.zip` (12GB) on Google Drive (see `data_download_VQA.ipynb`).
Strategy: run `resnet50_concat` first and submit early (safety net), then rerun with
`vit_bert_attn` for the improved score. Only the last submission is graded and it must
stay >= 49.9%.

In [ ]:
# 1. Materialize the project files (src/ + configs/) — single source of truth, embedded.
import base64, os, json
FILES = json.loads(r'''{"src/__init__.py": "", "src/config.py": "IiIiTWluaW1hbCBjb25maWcgbG9hZGluZzogcmVhZCBhIFlBTUwgZmlsZSBpbnRvIGEgbmVzdGVkIGRpY3QgYWNjZXNzaWJsZSBieSBhdHRyaWJ1dGUuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgeWFtbAoKCmNsYXNzIENvbmZpZyhkaWN0KToKICAgICIiIkEgZGljdCB0aGF0IGFsc28gc3VwcG9ydHMgYXR0cmlidXRlIGFjY2VzcyAoY2ZnLnRyYWluLmVwb2NocykuIiIiCgogICAgZGVmIF9fZ2V0YXR0cl9fKHNlbGYsIGtleTogc3RyKSAtPiBBbnk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB2YWx1ZSA9IHNlbGZba2V5XQogICAgICAgIGV4Y2VwdCBLZXlFcnJvciBhcyBleGM6CiAgICAgICAgICAgIHJhaXNlIEF0dHJpYnV0ZUVycm9yKGtleSkgZnJvbSBleGMKICAgICAgICByZXR1cm4gQ29uZmlnKHZhbHVlKSBpZiBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KSBlbHNlIHZhbHVlCgoKZGVmIGxvYWRfY29uZmlnKHBhdGg6IHN0ciB8IFBhdGgpIC0+IENvbmZpZzoKICAgIHdpdGggb3BlbihwYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgcmV0dXJuIENvbmZpZyh5YW1sLnNhZmVfbG9hZChmKSkK", "src/dataset.py": "IiIiVml6V2l6IFZRQSBkYXRhc2V0LgoKRGF0YSBsYXlvdXQgKG1hdGNoZXMgdGhlIG9mZmljaWFsIGNvdXJzZSBkaXN0cmlidXRpb24pOgogICAge3Jvb3R9L3RyYWluLmpzb24sIHtyb290fS92YWxpZC5qc29uICAgKHBhbmRhcy5yZWFkX2pzb247IGNvbHVtbnM6IGltYWdlLCBxdWVzdGlvbiwgYW5zd2VycykKICAgIHtyb290fS90cmFpbi88aW1hZ2U+LCB7cm9vdH0vdmFsaWQvPGltYWdlPgoKYHRyYWluLmpzb25gIHJvd3MgaGF2ZSBgYW5zd2Vyc2AgPSBsaXN0IG9mIDEwIGRpY3RzICh7ImFuc3dlciI6IHN0ciwgLi4ufSk7IGB2YWxpZC5qc29uYAoodGhlIGhlbGQtb3V0IHRlc3Qgc2V0KSBoYXMgbm8gYW5zd2Vycy4gVGhlIHF1ZXN0aW9uL2Fuc3dlciB2b2NhYnVsYXJpZXMgYXJlIGJ1aWx0IG9uIHRoZQp0cmFpbmluZyBzcGxpdCBhbmQgY29waWVkIHRvIHZhbC90ZXN0IHZpYSBgdXBkYXRlX2RpY3RgLgoKUXVlc3Rpb24gcmVwcmVzZW50YXRpb24gaXMgc2VsZWN0YWJsZToKLSB0ZXh0X21vZGU9Im9uZWhvdCI6IGEgZml4ZWQgKHZvY2FiKzEsKSBtdWx0aS1ob3QgdmVjdG9yIChiYXNlbGluZSBiZWhhdmlvdXIpLgotIHRleHRfbW9kZT0idG9rZW5zIjogYSBwYWRkZWQgKG1heF9xbGVuLCkgTG9uZ1RlbnNvciBvZiB0b2tlbiBpZHMgcGx1cyBhbiBhdHRlbnRpb24gbWFzaywKICBmb3IgdGhlIEdSVSAvIEJFUlQgZW5jb2RlcnMuIFVzZXMgYW4gaW50ZXJuYWwgd29yZCB2b2NhYiBmb3IgR1JVLCBvciBhIEhGIHRva2VuaXplciBmb3IgQkVSVC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBzdGF0aXN0aWNzIGltcG9ydCBtb2RlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcwppbXBvcnQgdG9yY2gKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldAoKZnJvbSAudGV4dHV0aWxzIGltcG9ydCBwcm9jZXNzX3RleHQKClBBRCwgVU5LID0gIjxwYWQ+IiwgIjx1bms+IgoKCmNsYXNzIFZpeldpelZRQShEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHJvb3Q6IHN0ciwKICAgICAgICBzcGxpdDogc3RyLAogICAgICAgIHRyYW5zZm9ybT1Ob25lLAogICAgICAgIGFuc3dlcjogYm9vbCA9IFRydWUsCiAgICAgICAgdGV4dF9tb2RlOiBzdHIgPSAib25laG90IiwKICAgICAgICBhbnN3ZXJfdm9jYWJfc2l6ZTogaW50IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgdG9rZW5pemVyPU5vbmUsCiAgICAgICAgbWF4X3FsZW46IGludCA9IDMyLAogICAgKToKICAgICAgICBzZWxmLnJvb3QgPSByb290CiAgICAgICAgc2VsZi5zcGxpdCA9IHNwbGl0CiAgICAgICAgc2VsZi50cmFuc2Zvcm0gPSB0cmFuc2Zvcm0KICAgICAgICBzZWxmLmFuc3dlciA9IGFuc3dlcgogICAgICAgIHNlbGYudGV4dF9tb2RlID0gdGV4dF9tb2RlCiAgICAgICAgc2VsZi5hbnN3ZXJfdm9jYWJfc2l6ZSA9IGFuc3dlcl92b2NhYl9zaXplCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSB0b2tlbml6ZXIgICMgSEYgdG9rZW5pemVyIHdoZW4gdGV4dF9tb2RlID09ICJ0b2tlbnMiIGFuZCBlbmNvZGVyIGlzIEJFUlQKICAgICAgICBzZWxmLm1heF9xbGVuID0gbWF4X3FsZW4KCiAgICAgICAgc2VsZi5pbWFnZV9kaXIgPSBmIntyb290fS97c3BsaXR9IgogICAgICAgIHNlbGYuZGYgPSBwYW5kYXMucmVhZF9qc29uKGYie3Jvb3R9L3tzcGxpdH0uanNvbiIpCgogICAgICAgICMgVm9jYWJzIChidWlsdCBoZXJlIGZvciB0aGUgdHJhaW5pbmcgc3BsaXQ7IGNvcGllZCBmb3IgdmFsL3Rlc3QgdmlhIHVwZGF0ZV9kaWN0KS4KICAgICAgICBzZWxmLnF1ZXN0aW9uMmlkeDogZGljdFtzdHIsIGludF0gPSB7fQogICAgICAgIHNlbGYuYW5zd2VyMmlkeDogZGljdFtzdHIsIGludF0gPSB7fQogICAgICAgIHNlbGYuaWR4MmFuc3dlcjogZGljdFtpbnQsIHN0cl0gPSB7fQogICAgICAgICMgR1JVIHdvcmQgdm9jYWIgKDAgPSBwYWQsIGxhc3QgPSB1bmspOyBvbmx5IHVzZWQgZm9yIHRleHRfbW9kZT0idG9rZW5zIiB3aXRob3V0IGEgdG9rZW5pemVyLgogICAgICAgIHNlbGYud29yZDJpZHg6IGRpY3Rbc3RyLCBpbnRdID0ge1BBRDogMH0KCiAgICAgICAgc2VsZi5fYnVpbGRfcXVlc3Rpb25fdm9jYWIoKQogICAgICAgIGlmIHNlbGYuYW5zd2VyOgogICAgICAgICAgICBzZWxmLl9idWlsZF9hbnN3ZXJfdm9jYWIoKQoKICAgICMgLS0gdm9jYWIgY29uc3RydWN0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfYnVpbGRfcXVlc3Rpb25fdm9jYWIoc2VsZikgLT4gTm9uZToKICAgICAgICBmb3IgcXVlc3Rpb24gaW4gc2VsZi5kZlsicXVlc3Rpb24iXToKICAgICAgICAgICAgZm9yIHdvcmQgaW4gcHJvY2Vzc190ZXh0KHF1ZXN0aW9uKS5zcGxpdCgiICIpOgogICAgICAgICAgICAgICAgaWYgd29yZCBhbmQgd29yZCBub3QgaW4gc2VsZi5xdWVzdGlvbjJpZHg6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5xdWVzdGlvbjJpZHhbd29yZF0gPSBsZW4oc2VsZi5xdWVzdGlvbjJpZHgpCiAgICAgICAgICAgICAgICBpZiB3b3JkIGFuZCB3b3JkIG5vdCBpbiBzZWxmLndvcmQyaWR4OgogICAgICAgICAgICAgICAgICAgIHNlbGYud29yZDJpZHhbd29yZF0gPSBsZW4oc2VsZi53b3JkMmlkeCkKICAgICAgICBzZWxmLndvcmQyaWR4W1VOS10gPSBsZW4oc2VsZi53b3JkMmlkeCkKCiAgICBkZWYgX2J1aWxkX2Fuc3dlcl92b2NhYihzZWxmKSAtPiBOb25lOgogICAgICAgIGNvdW50ZXI6IENvdW50ZXJbc3RyXSA9IENvdW50ZXIoKQogICAgICAgIGZvciBhbnN3ZXJzIGluIHNlbGYuZGZbImFuc3dlcnMiXToKICAgICAgICAgICAgZm9yIGEgaW4gYW5zd2VyczoKICAgICAgICAgICAgICAgIGNvdW50ZXJbcHJvY2Vzc190ZXh0KGFbImFuc3dlciJdKV0gKz0gMQoKICAgICAgICBpZiBzZWxmLmFuc3dlcl92b2NhYl9zaXplIGFuZCBzZWxmLmFuc3dlcl92b2NhYl9zaXplIDwgbGVuKGNvdW50ZXIpOgogICAgICAgICAgICBrZXB0ID0gW3cgZm9yIHcsIF8gaW4gY291bnRlci5tb3N0X2NvbW1vbihzZWxmLmFuc3dlcl92b2NhYl9zaXplKV0KICAgICAgICBlbHNlOgogICAgICAgICAgICBrZXB0ID0gbGlzdChjb3VudGVyLmtleXMoKSkKCiAgICAgICAgc2VsZi5hbnN3ZXIyaWR4ID0ge3c6IGkgZm9yIGksIHcgaW4gZW51bWVyYXRlKGtlcHQpfQogICAgICAgICMgUmVzZXJ2ZSBhIHRyYWlsaW5nIGluZGV4IGZvciBvdXQtb2Ytdm9jYWIgYW5zd2VycyAodG9wLU4gY2FzZSkuCiAgICAgICAgaWYgVU5LIG5vdCBpbiBzZWxmLmFuc3dlcjJpZHg6CiAgICAgICAgICAgIHNlbGYuYW5zd2VyMmlkeFtVTktdID0gbGVuKHNlbGYuYW5zd2VyMmlkeCkKICAgICAgICBzZWxmLmlkeDJhbnN3ZXIgPSB7aTogdyBmb3IgdywgaSBpbiBzZWxmLmFuc3dlcjJpZHguaXRlbXMoKX0KCiAgICBkZWYgdXBkYXRlX2RpY3Qoc2VsZiwgdHJhaW5fZGF0YXNldDogIlZpeldpelZRQSIpIC0+IE5vbmU6CiAgICAgICAgIiIiQ29weSB0aGUgdHJhaW5pbmctc3BsaXQgdm9jYWJ1bGFyaWVzIG9udG8gYSB2YWwvdGVzdCBkYXRhc2V0LiIiIgogICAgICAgIHNlbGYucXVlc3Rpb24yaWR4ID0gdHJhaW5fZGF0YXNldC5xdWVzdGlvbjJpZHgKICAgICAgICBzZWxmLmFuc3dlcjJpZHggPSB0cmFpbl9kYXRhc2V0LmFuc3dlcjJpZHgKICAgICAgICBzZWxmLmlkeDJhbnN3ZXIgPSB0cmFpbl9kYXRhc2V0LmlkeDJhbnN3ZXIKICAgICAgICBzZWxmLndvcmQyaWR4ID0gdHJhaW5fZGF0YXNldC53b3JkMmlkeAoKICAgICMgLS0gc2l6ZXMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEBwcm9wZXJ0eQogICAgZGVmIG9uZWhvdF9kaW0oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5xdWVzdGlvbjJpZHgpICsgMSAgIyArMSBmb3IgdGhlIHVua25vd24td29yZCBzbG90CgogICAgQHByb3BlcnR5CiAgICBkZWYgbnVtX2Fuc3dlcnMoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5hbnN3ZXIyaWR4KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHdvcmRfdm9jYWJfc2l6ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLndvcmQyaWR4KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHVua19hbnN3ZXJfaWR4KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5hbnN3ZXIyaWR4W1VOS10KCiAgICAjIC0tIGVuY29kaW5nIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgX2VuY29kZV9xdWVzdGlvbihzZWxmLCBxdWVzdGlvbjogc3RyKToKICAgICAgICB3b3JkcyA9IHByb2Nlc3NfdGV4dChxdWVzdGlvbikuc3BsaXQoIiAiKQogICAgICAgIGlmIHNlbGYudGV4dF9tb2RlID09ICJvbmVob3QiOgogICAgICAgICAgICB2ZWMgPSBucC56ZXJvcyhzZWxmLm9uZWhvdF9kaW0sIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgICAgIGZvciB3IGluIHdvcmRzOgogICAgICAgICAgICAgICAgdmVjW3NlbGYucXVlc3Rpb24yaWR4LmdldCh3LCBzZWxmLm9uZWhvdF9kaW0gLSAxKV0gPSAxLjAKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLmZyb21fbnVtcHkodmVjKQogICAgICAgIGlmIHNlbGYudGV4dF9tb2RlID09ICJ0b2tlbnMiOgogICAgICAgICAgICBpZiBzZWxmLnRva2VuaXplciBpcyBub3QgTm9uZTogICMgQkVSVAogICAgICAgICAgICAgICAgZW5jID0gc2VsZi50b2tlbml6ZXIoCiAgICAgICAgICAgICAgICAgICAgcXVlc3Rpb24sCiAgICAgICAgICAgICAgICAgICAgcGFkZGluZz0ibWF4X2xlbmd0aCIsCiAgICAgICAgICAgICAgICAgICAgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgIG1heF9sZW5ndGg9c2VsZi5tYXhfcWxlbiwKICAgICAgICAgICAgICAgICAgICByZXR1cm5fdGVuc29ycz0icHQiLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcmV0dXJuIGVuY1siaW5wdXRfaWRzIl0uc3F1ZWV6ZSgwKSwgZW5jWyJhdHRlbnRpb25fbWFzayJdLnNxdWVlemUoMCkKICAgICAgICAgICAgIyBHUlU6IGludGVybmFsIHdvcmQgdm9jYWIKICAgICAgICAgICAgdW5rID0gc2VsZi53b3JkMmlkeFtVTktdCiAgICAgICAgICAgIGlkcyA9IFtzZWxmLndvcmQyaWR4LmdldCh3LCB1bmspIGZvciB3IGluIHdvcmRzIGlmIHddWzogc2VsZi5tYXhfcWxlbl0KICAgICAgICAgICAgbWFzayA9IFsxXSAqIGxlbihpZHMpCiAgICAgICAgICAgIHBhZF9uID0gc2VsZi5tYXhfcWxlbiAtIGxlbihpZHMpCiAgICAgICAgICAgIGlkcyA9IGlkcyArIFtzZWxmLndvcmQyaWR4W1BBRF1dICogcGFkX24KICAgICAgICAgICAgbWFzayA9IG1hc2sgKyBbMF0gKiBwYWRfbgogICAgICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKGlkcywgZHR5cGU9dG9yY2gubG9uZyksIHRvcmNoLnRlbnNvcihtYXNrLCBkdHlwZT10b3JjaC5sb25nKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIHRleHRfbW9kZToge3NlbGYudGV4dF9tb2RlfSIpCgogICAgZGVmIF9lbmNvZGVfYW5zd2VycyhzZWxmLCBhbnN3ZXJzKToKICAgICAgICB1bmsgPSBzZWxmLnVua19hbnN3ZXJfaWR4CiAgICAgICAgaWR4cyA9IFtzZWxmLmFuc3dlcjJpZHguZ2V0KHByb2Nlc3NfdGV4dChhWyJhbnN3ZXIiXSksIHVuaykgZm9yIGEgaW4gYW5zd2Vyc10KICAgICAgICByZXR1cm4gaWR4cwoKICAgICMgLS0gRGF0YXNldCBwcm90b2NvbCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZGYpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KToKICAgICAgICBpbWFnZSA9IEltYWdlLm9wZW4oZiJ7c2VsZi5pbWFnZV9kaXJ9L3tzZWxmLmRmWydpbWFnZSddW2lkeF19IikuY29udmVydCgiUkdCIikKICAgICAgICBpZiBzZWxmLnRyYW5zZm9ybSBpcyBub3QgTm9uZToKICAgICAgICAgICAgaW1hZ2UgPSBzZWxmLnRyYW5zZm9ybShpbWFnZSkKCiAgICAgICAgcXVlc3Rpb24gPSBzZWxmLl9lbmNvZGVfcXVlc3Rpb24oc2VsZi5kZlsicXVlc3Rpb24iXVtpZHhdKQoKICAgICAgICBpZiBub3Qgc2VsZi5hbnN3ZXI6CiAgICAgICAgICAgIHJldHVybiBpbWFnZSwgcXVlc3Rpb24KCiAgICAgICAgYW5zd2VyX2lkeHMgPSBzZWxmLl9lbmNvZGVfYW5zd2VycyhzZWxmLmRmWyJhbnN3ZXJzIl1baWR4XSkKICAgICAgICBtb2RlX2Fuc3dlcl9pZHggPSBtb2RlKGFuc3dlcl9pZHhzKQogICAgICAgIHJldHVybiBpbWFnZSwgcXVlc3Rpb24sIHRvcmNoLnRlbnNvcihhbnN3ZXJfaWR4cywgZHR5cGU9dG9yY2gubG9uZyksIGludChtb2RlX2Fuc3dlcl9pZHgpCgoKSU1BR0VORVRfTUVBTiA9ICgwLjQ4NSwgMC40NTYsIDAuNDA2KQpJTUFHRU5FVF9TVEQgPSAoMC4yMjksIDAuMjI0LCAwLjIyNSkKCgpkZWYgYnVpbGRfaW1hZ2VfdHJhbnNmb3JtKGltYWdlX3NpemU6IGludCwgcHJldHJhaW5lZDogYm9vbCwgdHJhaW46IGJvb2wsIGF1Z21lbnQ6IGJvb2wgPSBGYWxzZSk6CiAgICAiIiJCdWlsZCB0aGUgdG9yY2h2aXNpb24gaW1hZ2UgdHJhbnNmb3JtLgoKICAgIFByZXRyYWluZWQgYmFja2JvbmVzIGV4cGVjdCBJbWFnZU5ldCBub3JtYWxpemF0aW9uOyB0aGUgc2NyYXRjaCBiYXNlbGluZSBqdXN0IHNjYWxlcwogICAgdG8gWzAsIDFdLiBMaWdodCBhdWdtZW50YXRpb24gKGNyb3AgKyBmbGlwICsgaml0dGVyKSBpcyBlbmFibGVkIGZvciB0cmFpbmluZyB3aGVuIGFza2VkLgogICAgIiIiCiAgICBmcm9tIHRvcmNodmlzaW9uIGltcG9ydCB0cmFuc2Zvcm1zCgogICAgb3BzID0gW10KICAgIGlmIHRyYWluIGFuZCBhdWdtZW50OgogICAgICAgIG9wcyArPSBbCiAgICAgICAgICAgIHRyYW5zZm9ybXMuUmFuZG9tUmVzaXplZENyb3AoaW1hZ2Vfc2l6ZSwgc2NhbGU9KDAuNywgMS4wKSksCiAgICAgICAgICAgIHRyYW5zZm9ybXMuUmFuZG9tSG9yaXpvbnRhbEZsaXAoKSwKICAgICAgICAgICAgdHJhbnNmb3Jtcy5Db2xvckppdHRlcigwLjIsIDAuMiwgMC4yKSwKICAgICAgICBdCiAgICBlbHNlOgogICAgICAgIG9wcyArPSBbdHJhbnNmb3Jtcy5SZXNpemUoKGltYWdlX3NpemUsIGltYWdlX3NpemUpKV0KICAgIG9wcyArPSBbdHJhbnNmb3Jtcy5Ub1RlbnNvcigpXQogICAgaWYgcHJldHJhaW5lZDoKICAgICAgICBvcHMgKz0gW3RyYW5zZm9ybXMuTm9ybWFsaXplKElNQUdFTkVUX01FQU4sIElNQUdFTkVUX1NURCldCiAgICByZXR1cm4gdHJhbnNmb3Jtcy5Db21wb3NlKG9wcykKCgpkZWYgc29mdF90YXJnZXRfZnJvbV9hbnN3ZXJzKGFuc3dlcl9pZHg6IHRvcmNoLlRlbnNvciwgbnVtX2Fuc3dlcnM6IGludCkgLT4gdG9yY2guVGVuc29yOgogICAgIiIiQnVpbGQgc29mdCBsYWJlbCB0YXJnZXRzIGZyb20gdGhlIDEwIGFubm90YXRvciBhbnN3ZXIgaW5kaWNlcy4KCiAgICBVc2VzIHRoZSBWUUEtc3R5bGUgc2NvcmUgbWluKGNvdW50LzMsIDEpIHBlciBhbnN3ZXIsIG5vcm1hbGl6ZWQgdG8gYSBkaXN0cmlidXRpb24uCiAgICBgYW5zd2VyX2lkeGAgaXMgKEIsIDEwKTsgcmV0dXJucyAoQiwgbnVtX2Fuc3dlcnMpLgogICAgIiIiCiAgICBiID0gYW5zd2VyX2lkeC5zaXplKDApCiAgICB0YXJnZXQgPSB0b3JjaC56ZXJvcyhiLCBudW1fYW5zd2VycywgZGV2aWNlPWFuc3dlcl9pZHguZGV2aWNlKQogICAgZm9yIGkgaW4gcmFuZ2UoYik6CiAgICAgICAgY291bnRzID0gQ291bnRlcihpbnQoYSkgZm9yIGEgaW4gYW5zd2VyX2lkeFtpXSkKICAgICAgICBmb3IgYW5zLCBjIGluIGNvdW50cy5pdGVtcygpOgogICAgICAgICAgICB0YXJnZXRbaSwgYW5zXSA9IG1pbihjIC8gMywgMS4wKQogICAgdG90YWwgPSB0YXJnZXQuc3VtKGRpbT0xLCBrZWVwZGltPVRydWUpLmNsYW1wX21pbigxZS04KQogICAgcmV0dXJuIHRhcmdldCAvIHRvdGFsCg==", "src/encoders.py": "IiIiSW1hZ2UgYW5kIHRleHQgZW5jb2RlcnMuCgpFYWNoIGVuY29kZXIgaXMgYSBzd2FwcGFibGUgY29tcG9uZW50IHNlbGVjdGVkIGJ5IGNvbmZpZyBzbyB0aGF0IGFibGF0aW9ucwooc2NyYXRjaCB2cy4gcHJldHJhaW5lZCwgb25lLWhvdCB2cy4gQkVSVCkgYXJlIGp1c3QgY29uZmlnIGNoYW5nZXMuCgpVbmlmaWVkIGludGVyZmFjZTogZXZlcnkgZW5jb2RlcidzIGBgZm9yd2FyZGBgIHJldHVybnMgYGAoZmVhdHVyZXMsIG1hc2spYGAgd2hlcmUKYGBmZWF0dXJlc2BgIGlzIGBgKEIsIFQsIEQpYGAgKGEgdG9rZW4gLyByZWdpb24gc2VxdWVuY2UpIGFuZCBgYG1hc2tgYCBpcyBgYChCLCBUKWBgIHdpdGgKMSBmb3IgdmFsaWQgcG9zaXRpb25zIG9yIGBgTm9uZWBgIHdoZW4gZXZlcnkgcG9zaXRpb24gaXMgdmFsaWQuIFRoaXMgbGV0cyBib3RoIHRoZSBjb25jYXQKYW5kIHRoZSBjcm9zcy1hdHRlbnRpb24gZnVzaW9uIGNvbnN1bWUgYW55IGVuY29kZXIuIEVhY2ggZW5jb2RlciBhbHNvIGV4cG9zZXMgYGAub3V0X2RpbWBgLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGltYWdlCgpjbGFzcyBfVG9yY2h2aXNpb25CYWNrYm9uZShubi5Nb2R1bGUpOgogICAgIiIiV3JhcCBhIHRvcmNodmlzaW9uIFJlc05ldCwgcmV0dXJuaW5nIGl0cyAoQiwgSFcsIEMpIGZlYXR1cmUtbWFwIHRva2Vucy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCBwcmV0cmFpbmVkOiBib29sLCBmcmVlemU6IGJvb2wpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGltcG9ydCB0b3JjaHZpc2lvbgogICAgICAgIGZyb20gdG9yY2h2aXNpb24ubW9kZWxzIGltcG9ydCBSZXNOZXQxOF9XZWlnaHRzLCBSZXNOZXQ1MF9XZWlnaHRzCgogICAgICAgIGlmIG5hbWUgPT0gInJlc25ldDE4IjoKICAgICAgICAgICAgd2VpZ2h0cyA9IFJlc05ldDE4X1dlaWdodHMuSU1BR0VORVQxS19WMSBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZQogICAgICAgICAgICBuZXQgPSB0b3JjaHZpc2lvbi5tb2RlbHMucmVzbmV0MTgod2VpZ2h0cz13ZWlnaHRzKQogICAgICAgICAgICBzZWxmLm91dF9kaW0gPSA1MTIKICAgICAgICBlbGlmIG5hbWUgPT0gInJlc25ldDUwIjoKICAgICAgICAgICAgd2VpZ2h0cyA9IFJlc05ldDUwX1dlaWdodHMuSU1BR0VORVQxS19WMiBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZQogICAgICAgICAgICBuZXQgPSB0b3JjaHZpc2lvbi5tb2RlbHMucmVzbmV0NTAod2VpZ2h0cz13ZWlnaHRzKQogICAgICAgICAgICBzZWxmLm91dF9kaW0gPSAyMDQ4CiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuc3VwcG9ydGVkIHRvcmNodmlzaW9uIGJhY2tib25lOiB7bmFtZX0iKQoKICAgICAgICAjIERyb3AgdGhlIGdsb2JhbCBwb29sICsgZmMgc28gd2Uga2VlcCB0aGUgc3BhdGlhbCBmZWF0dXJlIG1hcC4KICAgICAgICBzZWxmLmJvZHkgPSBubi5TZXF1ZW50aWFsKCpsaXN0KG5ldC5jaGlsZHJlbigpKVs6LTJdKQogICAgICAgIGlmIGZyZWV6ZToKICAgICAgICAgICAgZm9yIHAgaW4gc2VsZi5ib2R5LnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgICAgIHAucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2U6IHRvcmNoLlRlbnNvcik6CiAgICAgICAgZmVhdCA9IHNlbGYuYm9keShpbWFnZSkgICAgICAgICAgICAjIChCLCBDLCBILCBXKQogICAgICAgIGIsIGMsIGgsIHcgPSBmZWF0LnNoYXBlCiAgICAgICAgZmVhdCA9IGZlYXQuZmxhdHRlbigyKS50cmFuc3Bvc2UoMSwgMikgICMgKEIsIEhXLCBDKQogICAgICAgIHJldHVybiBmZWF0LCBOb25lCgoKY2xhc3MgX1RpbW1CYWNrYm9uZShubi5Nb2R1bGUpOgogICAgIiIiV3JhcCBhIHRpbW0gbW9kZWwgKFZpVCAvIENvbnZOZVh0KSwgcmV0dXJuaW5nIChCLCBULCBEKSB0b2tlbnMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG5hbWU6IHN0ciwgcHJldHJhaW5lZDogYm9vbCwgZnJlZXplOiBib29sKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpbXBvcnQgdGltbQoKICAgICAgICBtb2RlbF9uYW1lID0gewogICAgICAgICAgICAidml0IjogInZpdF9iYXNlX3BhdGNoMTZfMjI0IiwKICAgICAgICAgICAgImNvbnZuZXh0IjogImNvbnZuZXh0X3RpbnkiLAogICAgICAgIH1bbmFtZV0KICAgICAgICBzZWxmLm1vZGVsID0gdGltbS5jcmVhdGVfbW9kZWwobW9kZWxfbmFtZSwgcHJldHJhaW5lZD1wcmV0cmFpbmVkLCBudW1fY2xhc3Nlcz0wKQogICAgICAgIHNlbGYub3V0X2RpbSA9IHNlbGYubW9kZWwubnVtX2ZlYXR1cmVzCiAgICAgICAgc2VsZi5pc192aXQgPSBuYW1lID09ICJ2aXQiCiAgICAgICAgaWYgZnJlZXplOgogICAgICAgICAgICBmb3IgcCBpbiBzZWxmLm1vZGVsLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgICAgIHAucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2U6IHRvcmNoLlRlbnNvcik6CiAgICAgICAgZmVhdCA9IHNlbGYubW9kZWwuZm9yd2FyZF9mZWF0dXJlcyhpbWFnZSkKICAgICAgICBpZiBmZWF0LmRpbSgpID09IDQ6ICAgICAgICAgICAgICAgICMgQ29udk5lWHQ6IChCLCBDLCBILCBXKQogICAgICAgICAgICBmZWF0ID0gZmVhdC5mbGF0dGVuKDIpLnRyYW5zcG9zZSgxLCAyKQogICAgICAgICMgVmlUIGFscmVhZHkgcmV0dXJucyAoQiwgVCwgRCkKICAgICAgICByZXR1cm4gZmVhdCwgTm9uZQoKCmRlZiBidWlsZF9pbWFnZV9lbmNvZGVyKGNmZykgLT4gbm4uTW9kdWxlOgogICAgIiIidHlwZTogcmVzbmV0MTggfCByZXNuZXQ1MCB8IHZpdCB8IGNvbnZuZXh0LiIiIgogICAgdCA9IGNmZy50eXBlCiAgICBwcmV0cmFpbmVkID0gYm9vbChjZmcuZ2V0KCJwcmV0cmFpbmVkIiwgRmFsc2UpKQogICAgZnJlZXplID0gYm9vbChjZmcuZ2V0KCJmcmVlemUiLCBGYWxzZSkpCiAgICBpZiB0IGluICgicmVzbmV0MTgiLCAicmVzbmV0NTAiKToKICAgICAgICByZXR1cm4gX1RvcmNodmlzaW9uQmFja2JvbmUodCwgcHJldHJhaW5lZCwgZnJlZXplKQogICAgaWYgdCBpbiAoInZpdCIsICJjb252bmV4dCIpOgogICAgICAgIHJldHVybiBfVGltbUJhY2tib25lKHQsIHByZXRyYWluZWQsIGZyZWV6ZSkKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIGltYWdlIGVuY29kZXIgdHlwZToge3R9IikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0ZXh0CgpjbGFzcyBPbmVIb3RUZXh0RW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiQmFzZWxpbmUgdGV4dCBlbmNvZGVyOiBwcm9qZWN0IGEgbXVsdGktaG90IHF1ZXN0aW9uIHZlY3RvciB0byBhIGRlbnNlIGZlYXR1cmUuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZvY2FiX3NpemU6IGludCwgb3V0X2RpbTogaW50ID0gNTEyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnByb2ogPSBubi5MaW5lYXIodm9jYWJfc2l6ZSwgb3V0X2RpbSkKICAgICAgICBzZWxmLm91dF9kaW0gPSBvdXRfZGltCgogICAgZGVmIGZvcndhcmQoc2VsZiwgcXVlc3Rpb246IHRvcmNoLlRlbnNvcik6CiAgICAgICAgIyBxdWVzdGlvbjogKEIsIHZvY2FiX3NpemUpIG11bHRpLWhvdCAtPiAoQiwgMSwgb3V0X2RpbSksIG5vIHBhZGRpbmcKICAgICAgICByZXR1cm4gc2VsZi5wcm9qKHF1ZXN0aW9uKS51bnNxdWVlemUoMSksIE5vbmUKCgpjbGFzcyBHUlVUZXh0RW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiV29yZCBlbWJlZGRpbmdzICsgKGJpKUdSVSBvdmVyIHRva2VuIGlkcy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgdm9jYWJfc2l6ZTogaW50LCBlbWJlZF9kaW06IGludCA9IDMwMCwgaGlkZGVuOiBpbnQgPSA1MTIsCiAgICAgICAgICAgICAgICAgYmlkaXJlY3Rpb25hbDogYm9vbCA9IFRydWUsIHBhZF9pZHg6IGludCA9IDApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZW1iZWQgPSBubi5FbWJlZGRpbmcodm9jYWJfc2l6ZSwgZW1iZWRfZGltLCBwYWRkaW5nX2lkeD1wYWRfaWR4KQogICAgICAgIHNlbGYuZ3J1ID0gbm4uR1JVKGVtYmVkX2RpbSwgaGlkZGVuLCBiYXRjaF9maXJzdD1UcnVlLCBiaWRpcmVjdGlvbmFsPWJpZGlyZWN0aW9uYWwpCiAgICAgICAgc2VsZi5vdXRfZGltID0gaGlkZGVuICogKDIgaWYgYmlkaXJlY3Rpb25hbCBlbHNlIDEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgcXVlc3Rpb24pOgogICAgICAgIGlkcywgbWFzayA9IHF1ZXN0aW9uICAgICAgICAgICAgICAgIyAoQiwgTCksIChCLCBMKQogICAgICAgIGVtYiA9IHNlbGYuZW1iZWQoaWRzKQogICAgICAgIG91dCwgXyA9IHNlbGYuZ3J1KGVtYikgICAgICAgICAgICAgIyAoQiwgTCwgb3V0X2RpbSkKICAgICAgICByZXR1cm4gb3V0LCBtYXNrCgoKY2xhc3MgQmVydFRleHRFbmNvZGVyKG5uLk1vZHVsZSk6CiAgICAiIiJQcmV0cmFpbmVkIEJFUlQgdXNlZCBhcyBhIGNvbXBvbmVudDsgZmluZS10dW5lZCBieSBvdXIgdHJhaW5pbmcgbG9vcC4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyID0gImJlcnQtYmFzZS11bmNhc2VkIiwgZnJlZXplOiBib29sID0gRmFsc2UpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwKCiAgICAgICAgc2VsZi5iZXJ0ID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZChuYW1lKQogICAgICAgIHNlbGYub3V0X2RpbSA9IHNlbGYuYmVydC5jb25maWcuaGlkZGVuX3NpemUKICAgICAgICBpZiBmcmVlemU6CiAgICAgICAgICAgIGZvciBwIGluIHNlbGYuYmVydC5wYXJhbWV0ZXJzKCk6CiAgICAgICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHF1ZXN0aW9uKToKICAgICAgICBpZHMsIG1hc2sgPSBxdWVzdGlvbiAgICAgICAgICAgICAgICMgKEIsIEwpLCAoQiwgTCkKICAgICAgICBvdXQgPSBzZWxmLmJlcnQoaW5wdXRfaWRzPWlkcywgYXR0ZW50aW9uX21hc2s9bWFzaykubGFzdF9oaWRkZW5fc3RhdGUKICAgICAgICByZXR1cm4gb3V0LCBtYXNrCgoKZGVmIGJ1aWxkX3RleHRfZW5jb2RlcihjZmcsIG9uZWhvdF9kaW06IGludCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgIHdvcmRfdm9jYWJfc2l6ZTogaW50IHwgTm9uZSA9IE5vbmUpIC0+IG5uLk1vZHVsZToKICAgICIiInR5cGU6IG9uZWhvdCB8IGdydSB8IGJlcnQuCgogICAgYG9uZWhvdF9kaW1gIChvbmUtaG90IHZvY2FiIHNpemUpIGlzIHJlcXVpcmVkIGZvciB0aGUgb25laG90IGVuY29kZXI7CiAgICBgd29yZF92b2NhYl9zaXplYCBpcyByZXF1aXJlZCBmb3IgdGhlIEdSVSBlbmNvZGVyLgogICAgIiIiCiAgICB0ID0gY2ZnLnR5cGUKICAgIGlmIHQgPT0gIm9uZWhvdCI6CiAgICAgICAgYXNzZXJ0IG9uZWhvdF9kaW0gaXMgbm90IE5vbmUsICJvbmVob3QgZW5jb2RlciBuZWVkcyBvbmVob3RfZGltIgogICAgICAgIHJldHVybiBPbmVIb3RUZXh0RW5jb2RlcihvbmVob3RfZGltLCBvdXRfZGltPWludChjZmcuZ2V0KCJvdXRfZGltIiwgNTEyKSkpCiAgICBpZiB0ID09ICJncnUiOgogICAgICAgIGFzc2VydCB3b3JkX3ZvY2FiX3NpemUgaXMgbm90IE5vbmUsICJncnUgZW5jb2RlciBuZWVkcyB3b3JkX3ZvY2FiX3NpemUiCiAgICAgICAgcmV0dXJuIEdSVVRleHRFbmNvZGVyKAogICAgICAgICAgICB3b3JkX3ZvY2FiX3NpemUsCiAgICAgICAgICAgIGVtYmVkX2RpbT1pbnQoY2ZnLmdldCgiZW1iZWRfZGltIiwgMzAwKSksCiAgICAgICAgICAgIGhpZGRlbj1pbnQoY2ZnLmdldCgiaGlkZGVuIiwgNTEyKSksCiAgICAgICAgICAgIGJpZGlyZWN0aW9uYWw9Ym9vbChjZmcuZ2V0KCJiaWRpcmVjdGlvbmFsIiwgVHJ1ZSkpLAogICAgICAgICkKICAgIGlmIHQgPT0gImJlcnQiOgogICAgICAgIHJldHVybiBCZXJ0VGV4dEVuY29kZXIoCiAgICAgICAgICAgIG5hbWU9c3RyKGNmZy5nZXQoIm5hbWUiLCAiYmVydC1iYXNlLXVuY2FzZWQiKSksCiAgICAgICAgICAgIGZyZWV6ZT1ib29sKGNmZy5nZXQoImZyZWV6ZSIsIEZhbHNlKSksCiAgICAgICAgKQogICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gdGV4dCBlbmNvZGVyIHR5cGU6IHt0fSIpCg==", "src/metrics.py": "IiIiVlFBIGFjY3VyYWN5IG1ldHJpYyAoVml6V2l6IC8gVlFBIHYyIGNvbnZlbnRpb24pLgoKYWNjKGFuc3dlcikgPSBtaW4oI2h1bWFucyB0aGF0IGdhdmUgdGhhdCBhbnN3ZXIgLyAzLCAxKSwgYXZlcmFnZWQgb3ZlciB0aGUgMTAKbGVhdmUtb25lLW91dCBzdWJzZXRzLiBTZWUgaHR0cHM6Ly92aXN1YWxxYS5vcmcvZXZhbHVhdGlvbi5odG1sCgpUd28gZW50cnkgcG9pbnRzOgotIGB2cWFfYWNjdXJhY3kocHJlZCwgZ3RfYW5zd2VycylgOiBzaW5nbGUgc2FtcGxlLCBvcGVyYXRlcyBvbiBub3JtYWxpemVkIHN0cmluZ3MuCi0gYHZxYV9hY2N1cmFjeV9iYXRjaChwcmVkX2lkeCwgYW5zd2VyX2lkeClgOiBiYXRjaGVkLCBvcGVyYXRlcyBvbiBhbnN3ZXIgaW5kaWNlcyBhbmQKICBtYXRjaGVzIHRoZSBvZmZpY2lhbCBiYXNlbGluZSdzIGBWUUFfY3JpdGVyaW9uYCBleGFjdGx5ICh1c2VkIGluc2lkZSB0aGUgdHJhaW4gbG9vcCkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5cGluZyBpbXBvcnQgU2VxdWVuY2UKCmZyb20gLnRleHR1dGlscyBpbXBvcnQgcHJvY2Vzc190ZXh0CgoKZGVmIHZxYV9hY2N1cmFjeShwcmVkOiBzdHIsIGd0X2Fuc3dlcnM6IFNlcXVlbmNlW3N0cl0pIC0+IGZsb2F0OgogICAgIiIiVlFBIGFjY3VyYWN5IGZvciBvbmUgcHJlZGljdGlvbiBhZ2FpbnN0IHRoZSAxMCBhbm5vdGF0b3IgYW5zd2VycyAoc3RyaW5ncykuCgogICAgQm90aCBgcHJlZGAgYW5kIGBndF9hbnN3ZXJzYCBhcmUgbm9ybWFsaXplZCB3aXRoIGBwcm9jZXNzX3RleHRgIGJlZm9yZSBjb21wYXJpc29uLAogICAgdGhlbiB0aGUgbGVhdmUtb25lLW91dCBmb3JtdWxhIGlzIGF2ZXJhZ2VkIG92ZXIgdGhlIDEwIGhlbGQtb3V0IHN1YnNldHMuCiAgICAiIiIKICAgIHByZWRfbiA9IHByb2Nlc3NfdGV4dChwcmVkKQogICAgZ3RzID0gW3Byb2Nlc3NfdGV4dChhKSBmb3IgYSBpbiBndF9hbnN3ZXJzXQogICAgbiA9IGxlbihndHMpCgogICAgdG90YWwgPSAwLjAKICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgIG1hdGNoID0gMAogICAgICAgIGZvciBqIGluIHJhbmdlKG4pOgogICAgICAgICAgICBpZiBpID09IGo6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBwcmVkX24gPT0gZ3RzW2pdOgogICAgICAgICAgICAgICAgbWF0Y2ggKz0gMQogICAgICAgIHRvdGFsICs9IG1pbihtYXRjaCAvIDMsIDEpCiAgICByZXR1cm4gdG90YWwgLyBuCgoKZGVmIHZxYV9hY2N1cmFjeV9iYXRjaChwcmVkX2lkeCwgYW5zd2VyX2lkeCkgLT4gZmxvYXQ6CiAgICAiIiJCYXRjaGVkIFZRQSBhY2N1cmFjeSBvdmVyIGFuc3dlciAqaW5kaWNlcyogKG1hdGNoZXMgb2ZmaWNpYWwgYFZRQV9jcml0ZXJpb25gKS4KCiAgICBQYXJhbWV0ZXJzCiAgICAtLS0tLS0tLS0tCiAgICBwcmVkX2lkeCA6IFRlbnNvciAoQiwpCiAgICAgICAgUHJlZGljdGVkIGFuc3dlciBpbmRleCBwZXIgc2FtcGxlIChlLmcuIGBgbG9naXRzLmFyZ21heCgxKWBgKS4KICAgIGFuc3dlcl9pZHggOiBUZW5zb3IgKEIsIDEwKQogICAgICAgIFRoZSAxMCBhbm5vdGF0b3IgYW5zd2VycyBhcyB2b2NhYnVsYXJ5IGluZGljZXMuCgogICAgUmV0dXJucyB0aGUgbWVhbiBWUUEgYWNjdXJhY3kgb3ZlciB0aGUgYmF0Y2guCiAgICAiIiIKICAgIHRvdGFsID0gMC4wCiAgICBiYXRjaCA9IGxlbihwcmVkX2lkeCkKICAgIGZvciBwcmVkLCBhbnN3ZXJzIGluIHppcChwcmVkX2lkeCwgYW5zd2VyX2lkeCk6CiAgICAgICAgcHJlZCA9IGludChwcmVkKQogICAgICAgIGFuc3dlcnMgPSBbaW50KGEpIGZvciBhIGluIGFuc3dlcnNdCiAgICAgICAgYWNjID0gMC4wCiAgICAgICAgbiA9IGxlbihhbnN3ZXJzKQogICAgICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgICAgICBtYXRjaCA9IDAKICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2Uobik6CiAgICAgICAgICAgICAgICBpZiBpID09IGo6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIHByZWQgPT0gYW5zd2Vyc1tqXToKICAgICAgICAgICAgICAgICAgICBtYXRjaCArPSAxCiAgICAgICAgICAgIGFjYyArPSBtaW4obWF0Y2ggLyAzLCAxKQogICAgICAgIHRvdGFsICs9IGFjYyAvIG4KICAgIHJldHVybiB0b3RhbCAvIGJhdGNoCg==", "src/model.py": "IiIiVGhlIGN1c3RvbSBWUUEgbW9kZWw6IGltYWdlIGVuY29kZXIgKyB0ZXh0IGVuY29kZXIgKyBmdXNpb24gKyBhbnN3ZXIgY2xhc3NpZmllci4KClByZXRyYWluZWQgYmFja2JvbmVzIGFyZSB1c2VkIG9ubHkgYXMgY29tcG9uZW50cyBoZXJlIGFuZCBmaW5lLXR1bmVkIGJ5IHRoZSBjdXN0b20KdHJhaW5pbmcgbG9vcCBpbiBgdHJhaW4ucHlgIOKAlCBubyBvZmYtdGhlLXNoZWxmIFZRQSBtb2RlbCBpcyB1c2VkIGVuZC10by1lbmQuCgpFbmNvZGVycyByZXR1cm4gYGAoZmVhdHVyZXMsIG1hc2spYGAgd2l0aCBgYGZlYXR1cmVzYGAgc2hhcGVkIGBgKEIsIFQsIEQpYGAuIEZ1c2lvbgptb2R1bGVzIGNvbnN1bWUgdGhlIHR3byBgYChmZWF0dXJlcywgbWFzaylgYCBwYWlycyBhbmQgcmV0dXJuIGEgZnVzZWQgYGAoQiwgaGlkZGVuX2RpbSlgYAp2ZWN0b3IsIHdoaWNoIHRoZSBjbGFzc2lmaWVyIG1hcHMgdG8gYW5zd2VyIGxvZ2l0cy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCmZyb20gLmVuY29kZXJzIGltcG9ydCBidWlsZF9pbWFnZV9lbmNvZGVyLCBidWlsZF90ZXh0X2VuY29kZXIKCgpkZWYgbWFza2VkX21lYW4oZmVhdDogdG9yY2guVGVuc29yLCBtYXNrKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJNZWFuLXBvb2wgYGAoQiwgVCwgRClgYCBvdmVyIFQsIGhvbm91cmluZyBhIGBgKEIsIFQpYGAgdmFsaWRpdHkgbWFzayBpZiBnaXZlbi4iIiIKICAgIGlmIG1hc2sgaXMgTm9uZToKICAgICAgICByZXR1cm4gZmVhdC5tZWFuKGRpbT0xKQogICAgbSA9IG1hc2sudW5zcXVlZXplKC0xKS50byhmZWF0LmR0eXBlKSAgICAgICAgICAjIChCLCBULCAxKQogICAgc3VtbWVkID0gKGZlYXQgKiBtKS5zdW0oZGltPTEpCiAgICBjb3VudCA9IG0uc3VtKGRpbT0xKS5jbGFtcF9taW4oMWUtNikKICAgIHJldHVybiBzdW1tZWQgLyBjb3VudAoKCmNsYXNzIENvbmNhdEZ1c2lvbihubi5Nb2R1bGUpOgogICAgIiIiQmFzZWxpbmUgZnVzaW9uOiBtZWFuLXBvb2wgZWFjaCBtb2RhbGl0eSwgY29uY2F0ZW5hdGUsIHRoZW4gYW4gTUxQLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbWFnZV9kaW06IGludCwgdGV4dF9kaW06IGludCwgaGlkZGVuX2RpbTogaW50LCBkcm9wb3V0OiBmbG9hdCA9IDAuMSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5tbHAgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoaW1hZ2VfZGltICsgdGV4dF9kaW0sIGhpZGRlbl9kaW0pLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3BvdXQoZHJvcG91dCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW5fZGltLCBoaWRkZW5fZGltKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZSwgdGV4dCk6CiAgICAgICAgaW1nX2ZlYXQsIGltZ19tYXNrID0gaW1hZ2UKICAgICAgICB0eHRfZmVhdCwgdHh0X21hc2sgPSB0ZXh0CiAgICAgICAgeCA9IHRvcmNoLmNhdChbbWFza2VkX21lYW4oaW1nX2ZlYXQsIGltZ19tYXNrKSwgbWFza2VkX21lYW4odHh0X2ZlYXQsIHR4dF9tYXNrKV0sIGRpbT0xKQogICAgICAgIHJldHVybiBzZWxmLm1scCh4KQoKCmNsYXNzIENyb3NzQXR0ZW50aW9uRnVzaW9uKG5uLk1vZHVsZSk6CiAgICAiIiJTZWxmLWRlc2lnbmVkIGNyb3NzLW1vZGFsIGF0dGVudGlvbjogcXVlc3Rpb24gdG9rZW5zIGF0dGVuZCBvdmVyIGltYWdlIHJlZ2lvbnMuCgogICAgQm90aCBtb2RhbGl0aWVzIGFyZSBwcm9qZWN0ZWQgdG8gYSBzaGFyZWQgd2lkdGg7IHRoZSBxdWVzdGlvbiBxdWVyaWVzIHRoZSBpbWFnZSB3aXRoCiAgICBtdWx0aS1oZWFkIGF0dGVudGlvbiAoYSByZXNpZHVhbCBrZWVwcyB0aGUgcmF3IHF1ZXN0aW9uIHNpZ25hbCksIGFuZCB0aGUgYXR0ZW5kZWQKICAgIHF1ZXN0aW9uIGNvbnRleHQgaXMgY29tYmluZWQgd2l0aCBhIHBvb2xlZCBpbWFnZSBzdW1tYXJ5LiBUaGlzIGlzIHRoZSByZXBvcnQgY2VudGVycGllY2UKICAgIGFuZCB0aGUgYWJsYXRpb24gY291bnRlcnBhcnQgdG8gYENvbmNhdEZ1c2lvbmAuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW1hZ2VfZGltOiBpbnQsIHRleHRfZGltOiBpbnQsIGhpZGRlbl9kaW06IGludCwKICAgICAgICAgICAgICAgICBudW1faGVhZHM6IGludCA9IDgsIGRyb3BvdXQ6IGZsb2F0ID0gMC4xKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmltZ19wcm9qID0gbm4uTGluZWFyKGltYWdlX2RpbSwgaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLnR4dF9wcm9qID0gbm4uTGluZWFyKHRleHRfZGltLCBoaWRkZW5fZGltKQogICAgICAgIHNlbGYuYXR0biA9IG5uLk11bHRpaGVhZEF0dGVudGlvbihoaWRkZW5fZGltLCBudW1faGVhZHMsIGRyb3BvdXQ9ZHJvcG91dCwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLm91dCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcigyICogaGlkZGVuX2RpbSwgaGlkZGVuX2RpbSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uRHJvcG91dChkcm9wb3V0KSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2UsIHRleHQpOgogICAgICAgIGltZ19mZWF0LCBpbWdfbWFzayA9IGltYWdlCiAgICAgICAgdHh0X2ZlYXQsIHR4dF9tYXNrID0gdGV4dAogICAgICAgIGltZ19wID0gc2VsZi5pbWdfcHJvaihpbWdfZmVhdCkgICAgICAgICAgICAgICAgICAgICMgKEIsIFRpLCBIKQogICAgICAgIHR4dF9wID0gc2VsZi50eHRfcHJvaih0eHRfZmVhdCkgICAgICAgICAgICAgICAgICAgICMgKEIsIFRxLCBIKQoKICAgICAgICBrZXlfcGFkZGluZyA9IE5vbmUgaWYgaW1nX21hc2sgaXMgTm9uZSBlbHNlIChpbWdfbWFzayA9PSAwKQogICAgICAgIGF0dGVuZGVkLCBfID0gc2VsZi5hdHRuKHR4dF9wLCBpbWdfcCwgaW1nX3AsIGtleV9wYWRkaW5nX21hc2s9a2V5X3BhZGRpbmcpCiAgICAgICAgdHh0X2N0eCA9IHNlbGYubm9ybSh0eHRfcCArIGF0dGVuZGVkKSAgICAgICAgICAgICAgIyByZXNpZHVhbCBvdmVyIHRoZSBxdWVzdGlvbiB0b2tlbnMKCiAgICAgICAgdHh0X3ZlYyA9IG1hc2tlZF9tZWFuKHR4dF9jdHgsIHR4dF9tYXNrKSAgICAgICAgICAgIyAoQiwgSCkKICAgICAgICBpbWdfdmVjID0gbWFza2VkX21lYW4oaW1nX3AsIGltZ19tYXNrKSAgICAgICAgICAgICAjIChCLCBIKQogICAgICAgIHJldHVybiBzZWxmLm91dCh0b3JjaC5jYXQoW3R4dF92ZWMsIGltZ192ZWNdLCBkaW09MSkpCgoKZGVmIGJ1aWxkX2Z1c2lvbihjZmcsIGltYWdlX2RpbTogaW50LCB0ZXh0X2RpbTogaW50KSAtPiBubi5Nb2R1bGU6CiAgICBpZiBjZmcudHlwZSA9PSAiY29uY2F0IjoKICAgICAgICByZXR1cm4gQ29uY2F0RnVzaW9uKGltYWdlX2RpbSwgdGV4dF9kaW0sIGNmZy5oaWRkZW5fZGltKQogICAgaWYgY2ZnLnR5cGUgPT0gImNyb3NzX2F0dGVudGlvbiI6CiAgICAgICAgcmV0dXJuIENyb3NzQXR0ZW50aW9uRnVzaW9uKGltYWdlX2RpbSwgdGV4dF9kaW0sIGNmZy5oaWRkZW5fZGltKQogICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gZnVzaW9uIHR5cGU6IHtjZmcudHlwZX0iKQoKCmNsYXNzIFZRQU1vZGVsKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnLCBudW1fYW5zd2VyczogaW50LCBvbmVob3RfZGltOiBpbnQgfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgICAgICB3b3JkX3ZvY2FiX3NpemU6IGludCB8IE5vbmUgPSBOb25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmltYWdlX2VuY29kZXIgPSBidWlsZF9pbWFnZV9lbmNvZGVyKGNmZy5tb2RlbC5pbWFnZV9lbmNvZGVyKQogICAgICAgIHNlbGYudGV4dF9lbmNvZGVyID0gYnVpbGRfdGV4dF9lbmNvZGVyKAogICAgICAgICAgICBjZmcubW9kZWwudGV4dF9lbmNvZGVyLCBvbmVob3RfZGltPW9uZWhvdF9kaW0sIHdvcmRfdm9jYWJfc2l6ZT13b3JkX3ZvY2FiX3NpemUsCiAgICAgICAgKQogICAgICAgIHNlbGYuZnVzaW9uID0gYnVpbGRfZnVzaW9uKAogICAgICAgICAgICBjZmcubW9kZWwuZnVzaW9uLAogICAgICAgICAgICBzZWxmLmltYWdlX2VuY29kZXIub3V0X2RpbSwKICAgICAgICAgICAgc2VsZi50ZXh0X2VuY29kZXIub3V0X2RpbSwKICAgICAgICApCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uTGluZWFyKGNmZy5tb2RlbC5mdXNpb24uaGlkZGVuX2RpbSwgbnVtX2Fuc3dlcnMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2U6IHRvcmNoLlRlbnNvciwgcXVlc3Rpb24pIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpbWcgPSBzZWxmLmltYWdlX2VuY29kZXIoaW1hZ2UpICAgICAjIChmZWF0LCBtYXNrKQogICAgICAgIHR4dCA9IHNlbGYudGV4dF9lbmNvZGVyKHF1ZXN0aW9uKSAgICMgKGZlYXQsIG1hc2spCiAgICAgICAgZnVzZWQgPSBzZWxmLmZ1c2lvbihpbWcsIHR4dCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKGZ1c2VkKQo=", "src/predict.py": "IiIiSW5mZXJlbmNlIC8gc3VibWlzc2lvbiBlbnRyeSBwb2ludC4KClVzYWdlOgogICAgcHl0aG9uIC1tIHNyYy5wcmVkaWN0IC0tY29uZmlnIGNvbmZpZ3MvYmFzZWxpbmUueWFtbCAtLWNrcHQgZXhwZXJpbWVudHMvYmFzZWxpbmUvYmVzdC5wdAoKV3JpdGVzIGBzdWJtaXNzaW9uLm5weWAgKGEgbnVtcHkgYXJyYXkgb2YgYW5zd2VyICpzdHJpbmdzKiwgb25lIHBlciB0ZXN0IHNhbXBsZSwgaW4KYHZhbGlkLmpzb25gIG9yZGVyKSBwbHVzIGEgYG1vZGVsLnB0YCB3ZWlnaHRzIGZpbGUg4oCUIHRoZSB0d28gYXJ0aWZhY3RzIHRoZSBPbW5pY2FtcHVzIHppcApuZWVkcyBhbG9uZ3NpZGUgdGhlIG5vdGVib29rLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBvcwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKCmZyb20gLmNvbmZpZyBpbXBvcnQgbG9hZF9jb25maWcKZnJvbSAuZGF0YXNldCBpbXBvcnQgUEFELCBVTkssIFZpeldpelZRQSwgYnVpbGRfaW1hZ2VfdHJhbnNmb3JtCmZyb20gLm1vZGVsIGltcG9ydCBWUUFNb2RlbApmcm9tIC50cmFpbiBpbXBvcnQgYnVpbGRfdG9rZW5pemVyLCBwaWNrX2RldmljZSwgdG9fZGV2aWNlCgoKZGVmIHByZWRpY3QoY2ZnLCBja3B0X3BhdGg6IHN0ciwgb3V0OiBzdHIpIC0+IE5vbmU6CiAgICBkZXZpY2UgPSBwaWNrX2RldmljZSgpCiAgICBja3B0ID0gdG9yY2gubG9hZChja3B0X3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSkKCiAgICBpZHgyYW5zd2VyID0gY2twdFsiaWR4MmFuc3dlciJdCiAgICBxdWVzdGlvbjJpZHggPSBja3B0WyJxdWVzdGlvbjJpZHgiXQogICAgd29yZDJpZHggPSBja3B0WyJ3b3JkMmlkeCJdCgogICAgdGV4dF9tb2RlID0gIm9uZWhvdCIgaWYgY2ZnLm1vZGVsLnRleHRfZW5jb2Rlci50eXBlID09ICJvbmVob3QiIGVsc2UgInRva2VucyIKICAgIHByZXRyYWluZWQgPSBib29sKGNmZy5tb2RlbC5pbWFnZV9lbmNvZGVyLmdldCgicHJldHJhaW5lZCIsIEZhbHNlKSkKICAgIHRva2VuaXplciA9IGJ1aWxkX3Rva2VuaXplcihjZmcpCiAgICB0ZiA9IGJ1aWxkX2ltYWdlX3RyYW5zZm9ybShjZmcuZGF0YS5pbWFnZV9zaXplLCBwcmV0cmFpbmVkLCB0cmFpbj1GYWxzZSkKCiAgICAjIFRlc3Qgc3BsaXQgKCJ2YWxpZC5qc29uIik7IG92ZXJ3cml0ZSBpdHMgdm9jYWIgd2l0aCB0aGUgdHJhaW5pbmcgdm9jYWIgZnJvbSB0aGUgY2twdC4KICAgIHRlc3QgPSBWaXpXaXpWUUEoCiAgICAgICAgcm9vdD1jZmcuZGF0YS5yb290LCBzcGxpdD0idmFsaWQiLCB0cmFuc2Zvcm09dGYsIGFuc3dlcj1GYWxzZSwKICAgICAgICB0ZXh0X21vZGU9dGV4dF9tb2RlLCB0b2tlbml6ZXI9dG9rZW5pemVyLCBtYXhfcWxlbj1pbnQoY2ZnLmRhdGEuZ2V0KCJtYXhfcWxlbiIsIDMyKSksCiAgICApCiAgICB0ZXN0LnF1ZXN0aW9uMmlkeCA9IHF1ZXN0aW9uMmlkeAogICAgdGVzdC53b3JkMmlkeCA9IHdvcmQyaWR4CiAgICB0ZXN0LmFuc3dlcjJpZHggPSB7djogayBmb3IgaywgdiBpbiBpZHgyYW5zd2VyLml0ZW1zKCl9CiAgICB0ZXN0LmlkeDJhbnN3ZXIgPSBpZHgyYW5zd2VyCgogICAgbW9kZWwgPSBWUUFNb2RlbCgKICAgICAgICBjZmcsIG51bV9hbnN3ZXJzPWxlbihpZHgyYW5zd2VyKSwKICAgICAgICBvbmVob3RfZGltPWxlbihxdWVzdGlvbjJpZHgpICsgMSBpZiB0ZXh0X21vZGUgPT0gIm9uZWhvdCIgZWxzZSBOb25lLAogICAgICAgIHdvcmRfdm9jYWJfc2l6ZT1sZW4od29yZDJpZHgpIGlmIGNmZy5tb2RlbC50ZXh0X2VuY29kZXIudHlwZSA9PSAiZ3J1IiBlbHNlIE5vbmUsCiAgICApLnRvKGRldmljZSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChja3B0WyJtb2RlbF9zdGF0ZSJdKQogICAgbW9kZWwuZXZhbCgpCgogICAgbG9hZGVyID0gRGF0YUxvYWRlcih0ZXN0LCBiYXRjaF9zaXplPWludChjZmcudHJhaW4uZ2V0KCJiYXRjaF9zaXplIiwgNjQpKSwgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9aW50KGNmZy5kYXRhLmdldCgibnVtX3dvcmtlcnMiLCAyKSkpCgogICAgIyBOZXZlciBlbWl0IHRoZSBwbGFjZWhvbGRlciBjbGFzc2VzIGFzIGFuIGFuc3dlciBzdHJpbmcg4oCUIHRoZXkgYWx3YXlzIHNjb3JlIDAgb24gdGhlCiAgICAjIHJlYWwgdGVzdCAobm8gYW5ub3RhdG9yIGV2ZXIgYW5zd2VycyAiPHVuaz4iKS4gTWFzayB0aGVtIHNvIGFyZ21heCBwaWNrcyBhIHJlYWwgYW5zd2VyLgogICAgYmFubmVkID0gW2kgZm9yIGksIGEgaW4gaWR4MmFuc3dlci5pdGVtcygpIGlmIGEgaW4gKFVOSywgUEFEKV0KCiAgICBwcmVkczogbGlzdFtzdHJdID0gW10KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBpbWFnZSwgcXVlc3Rpb24gaW4gbG9hZGVyOgogICAgICAgICAgICBpbWFnZSwgcXVlc3Rpb24gPSBpbWFnZS50byhkZXZpY2UpLCB0b19kZXZpY2UocXVlc3Rpb24sIGRldmljZSkKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2UsIHF1ZXN0aW9uKQogICAgICAgICAgICBpZiBiYW5uZWQ6CiAgICAgICAgICAgICAgICBsb2dpdHNbOiwgYmFubmVkXSA9IGZsb2F0KCItaW5mIikKICAgICAgICAgICAgZm9yIGlkeCBpbiBsb2dpdHMuYXJnbWF4KDEpLmNwdSgpLnRvbGlzdCgpOgogICAgICAgICAgICAgICAgcHJlZHMuYXBwZW5kKGlkeDJhbnN3ZXJbaWR4XSkKCiAgICBvdXRfZGlyID0gb3MucGF0aC5kaXJuYW1lKG91dCkgb3IgIi4iCiAgICBvcy5tYWtlZGlycyhvdXRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgc3VibWlzc2lvbiA9IG5wLmFycmF5KHByZWRzKQogICAgbnAuc2F2ZShvdXQsIHN1Ym1pc3Npb24pCiAgICB0b3JjaC5zYXZlKG1vZGVsLnN0YXRlX2RpY3QoKSwgb3MucGF0aC5qb2luKG91dF9kaXIsICJtb2RlbC5wdCIpKQogICAgcHJpbnQoZiJ3cm90ZSB7b3V0fSAgc2hhcGU9e3N1Ym1pc3Npb24uc2hhcGV9IGR0eXBlPXtzdWJtaXNzaW9uLmR0eXBlfSIpCiAgICBwcmludChmIndyb3RlIHtvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ21vZGVsLnB0Jyl9IikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNrcHQiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PSJzdWJtaXNzaW9uL3N1Ym1pc3Npb24ubnB5IikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmVkaWN0KGxvYWRfY29uZmlnKGFyZ3MuY29uZmlnKSwgYXJncy5ja3B0LCBhcmdzLm91dCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "src/textutils.py": "IiIiVGV4dCBub3JtYWxpemF0aW9uIHNoYXJlZCBieSB0aGUgZGF0YXNldCBhbmQgdGhlIG1ldHJpYy4KCmBwcm9jZXNzX3RleHRgIGlzIHBvcnRlZCB2ZXJiYXRpbSBmcm9tIHRoZSBvZmZpY2lhbCBjb3Vyc2UgYmFzZWxpbmUgc28gdGhhdCBvdXIKYW5zd2VyIHZvY2FidWxhcnkgYW5kIGFjY3VyYWN5IGNvbXB1dGF0aW9uIG1hdGNoIHRoZSBncmFkZXIncyBub3JtYWxpemF0aW9uLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCgoKZGVmIHByb2Nlc3NfdGV4dCh0ZXh0OiBzdHIpIC0+IHN0cjoKICAgICIiIk5vcm1hbGl6ZSBhIHF1ZXN0aW9uIG9yIGFuc3dlciBzdHJpbmcgKGxvd2VyY2FzZSwgZGlnaXRzLCBkcm9wIGFydGljbGVzLCAuLi4pLgoKICAgIFBvcnRlZCBmcm9tIHRoZSBETCBCYXNpYyAyMDI2IFNwcmluZyBWUUEgYmFzZWxpbmUgbm90ZWJvb2suIEtlZXBpbmcgdGhpcyBpZGVudGljYWwKICAgIHRvIHRoZSBvZmZpY2lhbCB2ZXJzaW9uIGlzIGltcG9ydGFudDogdGhlIGFuc3dlciB2b2NhYnVsYXJ5IGFuZCB0aGUgVlFBIGFjY3VyYWN5IGFyZQogICAgYm90aCBkZWZpbmVkIG92ZXIgdGhlc2Ugbm9ybWFsaXplZCBzdHJpbmdzLgogICAgIiIiCiAgICAjIGxvd2VyY2FzZQogICAgdGV4dCA9IHRleHQubG93ZXIoKQoKICAgICMgY29udmVydCBudW1iZXIgd29yZHMgdG8gZGlnaXRzCiAgICBudW1fd29yZF90b19kaWdpdCA9IHsKICAgICAgICAiemVybyI6ICIwIiwgIm9uZSI6ICIxIiwgInR3byI6ICIyIiwgInRocmVlIjogIjMiLCAiZm91ciI6ICI0IiwKICAgICAgICAiZml2ZSI6ICI1IiwgInNpeCI6ICI2IiwgInNldmVuIjogIjciLCAiZWlnaHQiOiAiOCIsICJuaW5lIjogIjkiLAogICAgICAgICJ0ZW4iOiAiMTAiLAogICAgfQogICAgZm9yIHdvcmQsIGRpZ2l0IGluIG51bV93b3JkX3RvX2RpZ2l0Lml0ZW1zKCk6CiAgICAgICAgdGV4dCA9IHRleHQucmVwbGFjZSh3b3JkLCBkaWdpdCkKCiAgICAjIHJlbW92ZSBwZXJpb2RzIHRoYXQgYXJlIG5vdCBkZWNpbWFsIHBvaW50cwogICAgdGV4dCA9IHJlLnN1YihyIig/PCFcZClcLig/IVxkKSIsICIiLCB0ZXh0KQoKICAgICMgcmVtb3ZlIGFydGljbGVzCiAgICB0ZXh0ID0gcmUuc3ViKHIiXGIoYXxhbnx0aGUpXGIiLCAiIiwgdGV4dCkKCiAgICAjIG5vcm1hbGl6ZSBhIGZldyBjb250cmFjdGlvbnMKICAgIGNvbnRyYWN0aW9ucyA9IHsKICAgICAgICAiZG9udCI6ICJkb24ndCIsICJpc250IjogImlzbid0IiwgImFyZW50IjogImFyZW4ndCIsICJ3b250IjogIndvbid0IiwKICAgICAgICAiY2FudCI6ICJjYW4ndCIsICJ3b3VsZG50IjogIndvdWxkbid0IiwgImNvdWxkbnQiOiAiY291bGRuJ3QiLAogICAgfQogICAgZm9yIGNvbnRyYWN0aW9uLCBjb3JyZWN0IGluIGNvbnRyYWN0aW9ucy5pdGVtcygpOgogICAgICAgIHRleHQgPSB0ZXh0LnJlcGxhY2UoY29udHJhY3Rpb24sIGNvcnJlY3QpCgogICAgIyBwdW5jdHVhdGlvbiAtPiBzcGFjZQogICAgdGV4dCA9IHJlLnN1YihyIlteXHdccyc6XSIsICIgIiwgdGV4dCkKCiAgICAjIGNvbW1hIHNwYWNpbmcKICAgIHRleHQgPSByZS5zdWIociJccyssIiwgIiwiLCB0ZXh0KQoKICAgICMgY29sbGFwc2Ugd2hpdGVzcGFjZQogICAgdGV4dCA9IHJlLnN1YihyIlxzKyIsICIgIiwgdGV4dCkuc3RyaXAoKQoKICAgIHJldHVybiB0ZXh0Cg==", "src/train.py": "IiIiVHJhaW5pbmcgZW50cnkgcG9pbnQuIFVzYWdlOiBweXRob24gLW0gc3JjLnRyYWluIC0tY29uZmlnIGNvbmZpZ3MvYmFzZWxpbmUueWFtbAoKVHJhaW5zIHRoZSBjdXN0b20gYFZRQU1vZGVsYCBvbiB0aGUgZGlzdHJpYnV0ZWQgdHJhaW5pbmcgZGF0YS4gQSBzbGljZSBvZiB0aGUgdHJhaW5pbmcKc3BsaXQgaXMgaGVsZCBvdXQgYXMgYSBsb2NhbCB2YWxpZGF0aW9uIHNldCBzbyB3ZSBjYW4gZXN0aW1hdGUgdGhlIFZRQSBhY2N1cmFjeSAodGhlIHRlc3QKYW5zd2VycyBhcmUgbm90IGRpc3RyaWJ1dGVkKSBhbmQgYXZvaWQgd2FzdGluZyBPbW5pY2FtcHVzIHN1Ym1pc3Npb25zLiBUaGUgYmVzdCBjaGVja3BvaW50CnN0b3JlcyBgaWR4MmFuc3dlcmAsIHdoaWNoIGBwcmVkaWN0LnB5YCBuZWVkcyB0byB0dXJuIGxvZ2l0cyBiYWNrIGludG8gYW5zd2VyIHN0cmluZ3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgcmFuZG9tX3NwbGl0Cgpmcm9tIC5jb25maWcgaW1wb3J0IGxvYWRfY29uZmlnCmZyb20gLmRhdGFzZXQgaW1wb3J0IFBBRCwgVU5LLCBWaXpXaXpWUUEsIGJ1aWxkX2ltYWdlX3RyYW5zZm9ybSwgc29mdF90YXJnZXRfZnJvbV9hbnN3ZXJzCmZyb20gLm1ldHJpY3MgaW1wb3J0IHZxYV9hY2N1cmFjeV9iYXRjaApmcm9tIC5tb2RlbCBpbXBvcnQgVlFBTW9kZWwKCgpkZWYgcGlja19kZXZpY2UoKSAtPiBzdHI6CiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiAiY3VkYSIKICAgIGlmIHRvcmNoLmJhY2tlbmRzLm1wcy5pc19hdmFpbGFibGUoKToKICAgICAgICByZXR1cm4gIm1wcyIKICAgIHJldHVybiAiY3B1IgoKCmRlZiB0b19kZXZpY2UocXVlc3Rpb24sIGRldmljZSk6CiAgICAiIiJNb3ZlIGEgcXVlc3Rpb24gKG9uZS1ob3QgdGVuc29yLCBvciAoaWRzLCBtYXNrKSB0dXBsZSkgdG8gdGhlIGRldmljZS4iIiIKICAgIGlmIGlzaW5zdGFuY2UocXVlc3Rpb24sICh0dXBsZSwgbGlzdCkpOgogICAgICAgIHJldHVybiB0dXBsZShxLnRvKGRldmljZSkgZm9yIHEgaW4gcXVlc3Rpb24pCiAgICByZXR1cm4gcXVlc3Rpb24udG8oZGV2aWNlKQoKCmRlZiBidWlsZF90b2tlbml6ZXIoY2ZnKToKICAgICIiIkEgSEYgdG9rZW5pemVyIGlzIG9ubHkgbmVlZGVkIGZvciB0aGUgQkVSVCB0ZXh0IGVuY29kZXIuIiIiCiAgICBpZiBjZmcubW9kZWwudGV4dF9lbmNvZGVyLnR5cGUgPT0gImJlcnQiOgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyCgogICAgICAgIG5hbWUgPSBzdHIoY2ZnLm1vZGVsLnRleHRfZW5jb2Rlci5nZXQoIm5hbWUiLCAiYmVydC1iYXNlLXVuY2FzZWQiKSkKICAgICAgICByZXR1cm4gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQobmFtZSkKICAgIHJldHVybiBOb25lCgoKZGVmIG1ha2Vfc2NoZWR1bGVyKG9wdGltaXplciwgY2ZnLCBzdGVwc19wZXJfZXBvY2g6IGludCk6CiAgICB0b3RhbCA9IHN0ZXBzX3Blcl9lcG9jaCAqIGludChjZmcudHJhaW4uZXBvY2hzKQogICAgd2FybXVwID0gaW50KGNmZy50cmFpbi5nZXQoIndhcm11cF9zdGVwcyIsIDApKQogICAgaWYgY2ZnLnRyYWluLmdldCgic2NoZWR1bGVyIiwgIm5vbmUiKSAhPSAiY29zaW5lIjoKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBscl9sYW1iZGEoc3RlcDogaW50KSAtPiBmbG9hdDoKICAgICAgICBpZiB3YXJtdXAgPiAwIGFuZCBzdGVwIDwgd2FybXVwOgogICAgICAgICAgICByZXR1cm4gc3RlcCAvIG1heCgxLCB3YXJtdXApCiAgICAgICAgcHJvZ3Jlc3MgPSAoc3RlcCAtIHdhcm11cCkgLyBtYXgoMSwgdG90YWwgLSB3YXJtdXApCiAgICAgICAgcmV0dXJuIDAuNSAqICgxLjAgKyBtYXRoLmNvcyhtYXRoLnBpICogbWluKDEuMCwgcHJvZ3Jlc3MpKSkKCiAgICByZXR1cm4gdG9yY2gub3B0aW0ubHJfc2NoZWR1bGVyLkxhbWJkYUxSKG9wdGltaXplciwgbHJfbGFtYmRhKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlKG1vZGVsLCBsb2FkZXIsIGRldmljZSwgYmFubmVkPU5vbmUpIC0+IGZsb2F0OgogICAgbW9kZWwuZXZhbCgpCiAgICB0b3RhbCwgbiA9IDAuMCwgMAogICAgZm9yIGltYWdlLCBxdWVzdGlvbiwgYW5zd2VycywgXyBpbiBsb2FkZXI6CiAgICAgICAgaW1hZ2UsIHF1ZXN0aW9uID0gaW1hZ2UudG8oZGV2aWNlKSwgdG9fZGV2aWNlKHF1ZXN0aW9uLCBkZXZpY2UpCiAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2UsIHF1ZXN0aW9uKQogICAgICAgIGlmIGJhbm5lZDoKICAgICAgICAgICAgIyBNaXJyb3IgaW5mZXJlbmNlOiBwbGFjZWhvbGRlciBjbGFzc2VzIGFyZSBuZXZlciBlbWl0dGVkLCBzbyB0aGUgcmVwb3J0ZWQgdmFsCiAgICAgICAgICAgICMgYWNjdXJhY3kgcmVmbGVjdHMgdGhlIHJlYWwgKHN0cmluZy1tYXRjaGVkKSB0ZXN0IGJlaGF2aW91ciwgbm90IGFuIGluZmxhdGVkIHByb3h5LgogICAgICAgICAgICBsb2dpdHNbOiwgYmFubmVkXSA9IGZsb2F0KCItaW5mIikKICAgICAgICB0b3RhbCArPSB2cWFfYWNjdXJhY3lfYmF0Y2gobG9naXRzLmFyZ21heCgxKS5jcHUoKSwgYW5zd2VycykgKiBpbWFnZS5zaXplKDApCiAgICAgICAgbiArPSBpbWFnZS5zaXplKDApCiAgICByZXR1cm4gdG90YWwgLyBtYXgoMSwgbikKCgpkZWYgdHJhaW4oY2ZnKSAtPiBOb25lOgogICAgZGV2aWNlID0gcGlja19kZXZpY2UoKQogICAgdG9yY2gubWFudWFsX3NlZWQoaW50KGNmZy50cmFpbi5nZXQoInNlZWQiLCA0MikpKQogICAgdGV4dF9tb2RlID0gIm9uZWhvdCIgaWYgY2ZnLm1vZGVsLnRleHRfZW5jb2Rlci50eXBlID09ICJvbmVob3QiIGVsc2UgInRva2VucyIKICAgIHByZXRyYWluZWQgPSBib29sKGNmZy5tb2RlbC5pbWFnZV9lbmNvZGVyLmdldCgicHJldHJhaW5lZCIsIEZhbHNlKSkKICAgIHNvZnRfbGFiZWwgPSBib29sKGNmZy50cmFpbi5nZXQoInNvZnRfbGFiZWwiLCBGYWxzZSkpCgogICAgdG9rZW5pemVyID0gYnVpbGRfdG9rZW5pemVyKGNmZykKICAgIHRyYWluX3RmID0gYnVpbGRfaW1hZ2VfdHJhbnNmb3JtKGNmZy5kYXRhLmltYWdlX3NpemUsIHByZXRyYWluZWQsIHRyYWluPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhdWdtZW50PWJvb2woY2ZnLmRhdGEuZ2V0KCJhdWdtZW50IiwgRmFsc2UpKSkKICAgICMgRnVsbCB0cmFpbmluZyBkYXRhc2V0ICh2b2NhYiBidWlsdCBvdmVyIGFsbCB0cmFpbiByb3dzKSwgdGhlbiBzcGxpdCBpbnRvIHRyYWluL3ZhbC4KICAgIGZ1bGwgPSBWaXpXaXpWUUEoCiAgICAgICAgcm9vdD1jZmcuZGF0YS5yb290LCBzcGxpdD0idHJhaW4iLCB0cmFuc2Zvcm09dHJhaW5fdGYsIGFuc3dlcj1UcnVlLAogICAgICAgIHRleHRfbW9kZT10ZXh0X21vZGUsIGFuc3dlcl92b2NhYl9zaXplPWNmZy5kYXRhLmdldCgiYW5zd2VyX3ZvY2FiX3NpemUiKSwKICAgICAgICB0b2tlbml6ZXI9dG9rZW5pemVyLCBtYXhfcWxlbj1pbnQoY2ZnLmRhdGEuZ2V0KCJtYXhfcWxlbiIsIDMyKSksCiAgICApCiAgICB2YWxfZnJhY3Rpb24gPSBmbG9hdChjZmcuZGF0YS5nZXQoInZhbF9mcmFjdGlvbiIsIDAuMSkpCiAgICBuX3ZhbCA9IG1heCgxLCBpbnQobGVuKGZ1bGwpICogdmFsX2ZyYWN0aW9uKSkKICAgIG5fdHJhaW4gPSBsZW4oZnVsbCkgLSBuX3ZhbAogICAgZyA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKGludChjZmcudHJhaW4uZ2V0KCJzZWVkIiwgNDIpKSkKICAgIHRyYWluX3NldCwgdmFsX3NldCA9IHJhbmRvbV9zcGxpdChmdWxsLCBbbl90cmFpbiwgbl92YWxdLCBnZW5lcmF0b3I9ZykKCiAgICBudW1fd29ya2VycyA9IGludChjZmcuZGF0YS5nZXQoIm51bV93b3JrZXJzIiwgMikpCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluX3NldCwgYmF0Y2hfc2l6ZT1pbnQoY2ZnLnRyYWluLmJhdGNoX3NpemUpLCBzaHVmZmxlPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLCBwaW5fbWVtb3J5PShkZXZpY2UgPT0gImN1ZGEiKSkKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9zZXQsIGJhdGNoX3NpemU9aW50KGNmZy50cmFpbi5iYXRjaF9zaXplKSwgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLCBwaW5fbWVtb3J5PShkZXZpY2UgPT0gImN1ZGEiKSkKCiAgICBtb2RlbCA9IFZRQU1vZGVsKAogICAgICAgIGNmZywgbnVtX2Fuc3dlcnM9ZnVsbC5udW1fYW5zd2VycywKICAgICAgICBvbmVob3RfZGltPWZ1bGwub25laG90X2RpbSBpZiB0ZXh0X21vZGUgPT0gIm9uZWhvdCIgZWxzZSBOb25lLAogICAgICAgIHdvcmRfdm9jYWJfc2l6ZT1mdWxsLndvcmRfdm9jYWJfc2l6ZSBpZiBjZmcubW9kZWwudGV4dF9lbmNvZGVyLnR5cGUgPT0gImdydSIgZWxzZSBOb25lLAogICAgKS50byhkZXZpY2UpCgogICAgb3B0X25hbWUgPSBjZmcudHJhaW4uZ2V0KCJvcHRpbWl6ZXIiLCAiYWRhbSIpCiAgICBvcHRpbV9jbHMgPSB0b3JjaC5vcHRpbS5BZGFtVyBpZiBvcHRfbmFtZSA9PSAiYWRhbXciIGVsc2UgdG9yY2gub3B0aW0uQWRhbQogICAgb3B0aW1pemVyID0gb3B0aW1fY2xzKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9ZmxvYXQoY2ZnLnRyYWluLmxyKSwKICAgICAgICAgICAgICAgICAgICAgICAgICB3ZWlnaHRfZGVjYXk9ZmxvYXQoY2ZnLnRyYWluLmdldCgid2VpZ2h0X2RlY2F5IiwgMC4wKSkpCiAgICBzY2hlZHVsZXIgPSBtYWtlX3NjaGVkdWxlcihvcHRpbWl6ZXIsIGNmZywgc3RlcHNfcGVyX2Vwb2NoPWxlbih0cmFpbl9sb2FkZXIpKQogICAgdXNlX2FtcCA9IGJvb2woY2ZnLnRyYWluLmdldCgiYW1wIiwgRmFsc2UpKSBhbmQgZGV2aWNlID09ICJjdWRhIgogICAgc2NhbGVyID0gdG9yY2guY3VkYS5hbXAuR3JhZFNjYWxlcihlbmFibGVkPXVzZV9hbXApCgogICAgY2UgPSBubi5Dcm9zc0VudHJvcHlMb3NzKCkKICAgIGJhbm5lZCA9IFtpIGZvciBpLCBhIGluIGZ1bGwuaWR4MmFuc3dlci5pdGVtcygpIGlmIGEgaW4gKFVOSywgUEFEKV0KCiAgICBkZWYgY29tcHV0ZV9sb3NzKGxvZ2l0cywgYW5zd2VycywgbW9kZV9hbnN3ZXIpOgogICAgICAgIGlmIHNvZnRfbGFiZWw6CiAgICAgICAgICAgIHRhcmdldCA9IHNvZnRfdGFyZ2V0X2Zyb21fYW5zd2VycyhhbnN3ZXJzLnRvKGRldmljZSksIGZ1bGwubnVtX2Fuc3dlcnMpCiAgICAgICAgICAgIHJldHVybiAtKHRvcmNoLmxvZ19zb2Z0bWF4KGxvZ2l0cywgZGltPTEpICogdGFyZ2V0KS5zdW0oZGltPTEpLm1lYW4oKQogICAgICAgIHJldHVybiBjZShsb2dpdHMsIG1vZGVfYW5zd2VyLnRvKGRldmljZSkpCgogICAgb3MubWFrZWRpcnMoY2ZnLm91dHB1dF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICBiZXN0X2FjYywgaGlzdG9yeSA9IC0xLjAsIFtdCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKGludChjZmcudHJhaW4uZXBvY2hzKSk6CiAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgIHJ1bm5pbmcgPSAwLjAKICAgICAgICBmb3IgaW1hZ2UsIHF1ZXN0aW9uLCBhbnN3ZXJzLCBtb2RlX2Fuc3dlciBpbiB0cmFpbl9sb2FkZXI6CiAgICAgICAgICAgIGltYWdlLCBxdWVzdGlvbiA9IGltYWdlLnRvKGRldmljZSksIHRvX2RldmljZShxdWVzdGlvbiwgZGV2aWNlKQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgd2l0aCB0b3JjaC5jdWRhLmFtcC5hdXRvY2FzdChlbmFibGVkPXVzZV9hbXApOgogICAgICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaW1hZ2UsIHF1ZXN0aW9uKQogICAgICAgICAgICAgICAgbG9zcyA9IGNvbXB1dGVfbG9zcyhsb2dpdHMsIGFuc3dlcnMsIG1vZGVfYW5zd2VyKQogICAgICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgICAgICBzY2FsZXIuc3RlcChvcHRpbWl6ZXIpCiAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICBpZiBzY2hlZHVsZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIHJ1bm5pbmcgKz0gbG9zcy5pdGVtKCkKCiAgICAgICAgdmFsX2FjYyA9IGV2YWx1YXRlKG1vZGVsLCB2YWxfbG9hZGVyLCBkZXZpY2UsIGJhbm5lZD1iYW5uZWQpCiAgICAgICAgdHJhaW5fbG9zcyA9IHJ1bm5pbmcgLyBtYXgoMSwgbGVuKHRyYWluX2xvYWRlcikpCiAgICAgICAgaGlzdG9yeS5hcHBlbmQoeyJlcG9jaCI6IGVwb2NoICsgMSwgInRyYWluX2xvc3MiOiB0cmFpbl9sb3NzLCAidmFsX3ZxYV9hY2MiOiB2YWxfYWNjfSkKICAgICAgICBwcmludChmIlt7ZXBvY2ggKyAxfS97Y2ZnLnRyYWluLmVwb2Noc31dIHRyYWluX2xvc3M9e3RyYWluX2xvc3M6LjRmfSB2YWxfdnFhX2FjYz17dmFsX2FjYzouNGZ9IikKCiAgICAgICAgY2twdCA9IHsKICAgICAgICAgICAgIm1vZGVsX3N0YXRlIjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAiaWR4MmFuc3dlciI6IGZ1bGwuaWR4MmFuc3dlciwKICAgICAgICAgICAgInF1ZXN0aW9uMmlkeCI6IGZ1bGwucXVlc3Rpb24yaWR4LAogICAgICAgICAgICAid29yZDJpZHgiOiBmdWxsLndvcmQyaWR4LAogICAgICAgICAgICAiY29uZmlnIjogZGljdChjZmcpLAogICAgICAgICAgICAidmFsX3ZxYV9hY2MiOiB2YWxfYWNjLAogICAgICAgICAgICAiZXBvY2giOiBlcG9jaCArIDEsCiAgICAgICAgfQogICAgICAgIHRvcmNoLnNhdmUoY2twdCwgb3MucGF0aC5qb2luKGNmZy5vdXRwdXRfZGlyLCAibGFzdC5wdCIpKQogICAgICAgIGlmIHZhbF9hY2MgPiBiZXN0X2FjYzoKICAgICAgICAgICAgYmVzdF9hY2MgPSB2YWxfYWNjCiAgICAgICAgICAgIHRvcmNoLnNhdmUoY2twdCwgb3MucGF0aC5qb2luKGNmZy5vdXRwdXRfZGlyLCAiYmVzdC5wdCIpKQoKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oY2ZnLm91dHB1dF9kaXIsICJtZXRyaWNzLmpzb24iKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcCh7ImJlc3RfdmFsX3ZxYV9hY2MiOiBiZXN0X2FjYywgImhpc3RvcnkiOiBoaXN0b3J5fSwgZiwgaW5kZW50PTIpCiAgICBwcmludChmImRvbmUuIGJlc3RfdmFsX3ZxYV9hY2M9e2Jlc3RfYWNjOi40Zn0gICg+PSAwLjQ5OSB0YXJnZXQgZm9yIGNvbXBsZXRpb24pIikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdHJhaW4obG9hZF9jb25maWcoYXJncy5jb25maWcpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "configs/baseline.yaml": "IyBCYXNlbGluZSBleHBlcmltZW50IChtaXJyb3JzIHRoZSBwcm92aWRlZCB3ZWFrIGJhc2VsaW5lOiBzY3JhdGNoIFJlc05ldDE4ICsgb25lLWhvdCArIGNvbmNhdCkuCiMgU2VydmVzIGFzIHRoZSBsb3dlciBib3VuZCBmb3IgdGhlIHNjcmF0Y2gtdnMtcHJldHJhaW5lZCBhYmxhdGlvbi4KbmFtZTogYmFzZWxpbmUKCmRhdGE6CiAgcm9vdDogZGF0YQogIGltYWdlX3NpemU6IDIyNAogIG51bV93b3JrZXJzOiA0CiAgYW5zd2VyX3ZvY2FiX3NpemU6IG51bGwgICAjIG51bGwgPSBmdWxsIGFuc3dlciB2b2NhYnVsYXJ5IChtYXRjaGVzIHRoZSBvZmZpY2lhbCBiYXNlbGluZSkKICB2YWxfZnJhY3Rpb246IDAuMQogIG1heF9xbGVuOiAzMgogIGF1Z21lbnQ6IGZhbHNlCgptb2RlbDoKICBpbWFnZV9lbmNvZGVyOgogICAgdHlwZTogcmVzbmV0MTgKICAgIHByZXRyYWluZWQ6IGZhbHNlICAgICAgICAjIGJhc2VsaW5lIHRyYWlucyBmcm9tIHNjcmF0Y2gKICAgIGZyZWV6ZTogZmFsc2UKICB0ZXh0X2VuY29kZXI6CiAgICB0eXBlOiBvbmVob3QKICAgIG91dF9kaW06IDUxMgogIGZ1c2lvbjoKICAgIHR5cGU6IGNvbmNhdAogICAgaGlkZGVuX2RpbTogNTEyCgp0cmFpbjoKICBlcG9jaHM6IDQKICBiYXRjaF9zaXplOiAxMjgKICBscjogMS4wZS0zCiAgd2VpZ2h0X2RlY2F5OiAxLjBlLTUKICBvcHRpbWl6ZXI6IGFkYW0KICBzY2hlZHVsZXI6IG5vbmUKICB3YXJtdXBfc3RlcHM6IDAKICBhbXA6IGZhbHNlCiAgc29mdF9sYWJlbDogZmFsc2UKICBzZWVkOiA0MgoKb3V0cHV0X2RpcjogZXhwZXJpbWVudHMvYmFzZWxpbmUK", "configs/r50_bert_attn.yaml": "IyBSdW4gMiAobWFpbiBpbXByb3ZlZCBtb2RlbCk6IHByZXRyYWluZWQgUmVzTmV0NTAgKyBCRVJUICsgY3Jvc3MtYXR0ZW50aW9uICsgc29mdCBsYWJlbHMuCiMgTGlnaHRlciB0aGFuIFZpVCtCRVJUIHNvIGl0IGZpdHMgYSBDb2xhYiBUNCBpbiB0aW1lLCB3aGlsZSB1cGdyYWRpbmcgdGhlIGJvdHRsZW5lY2sgKHRleHQpCiMgYW5kIGFkZGluZyB0aGUgc2VsZi1kZXNpZ25lZCBjcm9zcy1tb2RhbCBhdHRlbnRpb24gdGhhdCBpcyB0aGUgcmVwb3J0IGNlbnRlcnBpZWNlLgpuYW1lOiByNTBfYmVydF9hdHRuCgpkYXRhOgogIHJvb3Q6IGRhdGEKICBpbWFnZV9zaXplOiAyMjQKICBudW1fd29ya2VyczogMgogIGFuc3dlcl92b2NhYl9zaXplOiBudWxsICAgIyBmdWxsIGFuc3dlciB2b2NhYnVsYXJ5IChubyBjYXRjaC1hbGwgPHVuaz4gY2xhc3MpCiAgdmFsX2ZyYWN0aW9uOiAwLjEKICBtYXhfcWxlbjogMzIKICBhdWdtZW50OiB0cnVlCgptb2RlbDoKICBpbWFnZV9lbmNvZGVyOgogICAgdHlwZTogcmVzbmV0NTAKICAgIHByZXRyYWluZWQ6IHRydWUKICAgIGZyZWV6ZTogZmFsc2UKICB0ZXh0X2VuY29kZXI6CiAgICB0eXBlOiBiZXJ0CiAgICBuYW1lOiBiZXJ0LWJhc2UtdW5jYXNlZAogICAgZnJlZXplOiBmYWxzZQogIGZ1c2lvbjoKICAgIHR5cGU6IGNyb3NzX2F0dGVudGlvbgogICAgaGlkZGVuX2RpbTogNzY4Cgp0cmFpbjoKICBlcG9jaHM6IDYKICBiYXRjaF9zaXplOiAxNiAgICAgICAgICAjIHNhZmUgZm9yIGEgMTZHQiBUNCB3aXRoIFJlc05ldDUwICsgQkVSVCBib3RoIGZpbmUtdHVuZWQgKyBBTVAKICBscjogMy4wZS01CiAgd2VpZ2h0X2RlY2F5OiAwLjAxCiAgb3B0aW1pemVyOiBhZGFtdwogIHNjaGVkdWxlcjogY29zaW5lCiAgd2FybXVwX3N0ZXBzOiA0MDAKICBhbXA6IHRydWUKICBzb2Z0X2xhYmVsOiB0cnVlCiAgc2VlZDogNDIKCm91dHB1dF9kaXI6IGV4cGVyaW1lbnRzL3I1MF9iZXJ0X2F0dG4K", "configs/resnet50_concat.yaml": "IyBTYWZldHktbmV0IHJ1biAoUnVuIDEpOiBwcmV0cmFpbmVkIFJlc05ldDUwICsgSW1hZ2VOZXQgbm9ybSArIG9uZS1ob3QgKyBjb25jYXQuCiMgRmFzdC1jb252ZXJnaW5nLCBsb3ctcmlzazsgZ29hbCBpcyBhIHZhbGlkIHN1Ym1pc3Npb24gY29tZm9ydGFibHkgYWJvdmUgdGhlIDQ5LjklIGxpbmUuCm5hbWU6IHJlc25ldDUwX2NvbmNhdAoKZGF0YToKICByb290OiBkYXRhCiAgaW1hZ2Vfc2l6ZTogMjI0CiAgbnVtX3dvcmtlcnM6IDIKICBhbnN3ZXJfdm9jYWJfc2l6ZTogbnVsbCAgICMgZnVsbCBhbnN3ZXIgdm9jYWJ1bGFyeSAobm8gY2F0Y2gtYWxsIDx1bms+IGNsYXNzKTsgbWF0Y2hlcyBiYXNlbGluZQogIHZhbF9mcmFjdGlvbjogMC4xCiAgbWF4X3FsZW46IDMyCiAgYXVnbWVudDogZmFsc2UKCm1vZGVsOgogIGltYWdlX2VuY29kZXI6CiAgICB0eXBlOiByZXNuZXQ1MAogICAgcHJldHJhaW5lZDogdHJ1ZQogICAgZnJlZXplOiBmYWxzZSAgICAgICAgICAgICMgZmluZS10dW5lIHRoZSBiYWNrYm9uZQogIHRleHRfZW5jb2RlcjoKICAgIHR5cGU6IG9uZWhvdAogICAgb3V0X2RpbTogNTEyCiAgZnVzaW9uOgogICAgdHlwZTogY29uY2F0CiAgICBoaWRkZW5fZGltOiAxMDI0Cgp0cmFpbjoKICBlcG9jaHM6IDYKICBiYXRjaF9zaXplOiA2NAogIGxyOiAxLjBlLTQKICB3ZWlnaHRfZGVjYXk6IDEuMGUtNAogIG9wdGltaXplcjogYWRhbXcKICBzY2hlZHVsZXI6IGNvc2luZQogIHdhcm11cF9zdGVwczogMzAwCiAgYW1wOiB0cnVlCiAgc29mdF9sYWJlbDogZmFsc2UKICBzZWVkOiA0MgoKb3V0cHV0X2RpcjogZXhwZXJpbWVudHMvcmVzbmV0NTBfY29uY2F0Cg==", "configs/smoke.yaml": "IyBUaW55IENQVSBzbW9rZSB0ZXN0IG92ZXIgc3ludGhldGljIGRhdGEgKHNlZSB0b29scy9tYWtlX3Ntb2tlX2RhdGEucHkpLiBOb3QgZm9yIHJlYWwgdHJhaW5pbmcuCm5hbWU6IHNtb2tlCgpkYXRhOgogIHJvb3Q6IGRhdGEvc21va2UKICBpbWFnZV9zaXplOiA2NAogIG51bV93b3JrZXJzOiAwCiAgYW5zd2VyX3ZvY2FiX3NpemU6IG51bGwKICB2YWxfZnJhY3Rpb246IDAuNQogIG1heF9xbGVuOiAxNgogIGF1Z21lbnQ6IGZhbHNlCgptb2RlbDoKICBpbWFnZV9lbmNvZGVyOgogICAgdHlwZTogcmVzbmV0MTgKICAgIHByZXRyYWluZWQ6IGZhbHNlCiAgICBmcmVlemU6IGZhbHNlCiAgdGV4dF9lbmNvZGVyOgogICAgdHlwZTogb25laG90CiAgICBvdXRfZGltOiA2NAogIGZ1c2lvbjoKICAgIHR5cGU6IGNvbmNhdAogICAgaGlkZGVuX2RpbTogNjQKCnRyYWluOgogIGVwb2NoczogMQogIGJhdGNoX3NpemU6IDQKICBscjogMS4wZS0zCiAgd2VpZ2h0X2RlY2F5OiAwLjAKICBvcHRpbWl6ZXI6IGFkYW0KICBzY2hlZHVsZXI6IG5vbmUKICB3YXJtdXBfc3RlcHM6IDAKICBhbXA6IGZhbHNlCiAgc29mdF9sYWJlbDogZmFsc2UKICBzZWVkOiAwCgpvdXRwdXRfZGlyOiBleHBlcmltZW50cy9zbW9rZQo=", "configs/vit_bert_attn.yaml": "IyBJbXByb3ZlZCBtb2RlbCAoUnVuIDIvMyk6IHByZXRyYWluZWQgVmlUICsgQkVSVCBlbmNvZGVycywgY3Jvc3MtbW9kYWwgYXR0ZW50aW9uLCBzb2Z0IGxhYmVscy4KIyBUaGUgY3Jvc3MtYXR0ZW50aW9uIGZ1c2lvbiBpcyB0aGUgcmVwb3J0IGNlbnRlcnBpZWNlOyB0b2dnbGluZyBmdXNpb24vZW5jb2Rlci9sYWJlbCBnaXZlcyB0aGUKIyBhYmxhdGlvbiB0YWJsZS4KbmFtZTogdml0X2JlcnRfYXR0bgoKZGF0YToKICByb290OiBkYXRhCiAgaW1hZ2Vfc2l6ZTogMjI0CiAgbnVtX3dvcmtlcnM6IDIKICBhbnN3ZXJfdm9jYWJfc2l6ZTogbnVsbCAgICMgZnVsbCBhbnN3ZXIgdm9jYWJ1bGFyeSAobm8gY2F0Y2gtYWxsIDx1bms+IGNsYXNzKQogIHZhbF9mcmFjdGlvbjogMC4xCiAgbWF4X3FsZW46IDMyCiAgYXVnbWVudDogdHJ1ZQoKbW9kZWw6CiAgaW1hZ2VfZW5jb2RlcjoKICAgIHR5cGU6IHZpdAogICAgcHJldHJhaW5lZDogdHJ1ZQogICAgZnJlZXplOiBmYWxzZSAgICAgICAgICAgICMgZmluZS10dW5lCiAgdGV4dF9lbmNvZGVyOgogICAgdHlwZTogYmVydAogICAgbmFtZTogYmVydC1iYXNlLXVuY2FzZWQKICAgIGZyZWV6ZTogZmFsc2UKICBmdXNpb246CiAgICB0eXBlOiBjcm9zc19hdHRlbnRpb24KICAgIGhpZGRlbl9kaW06IDc2OAoKdHJhaW46CiAgZXBvY2hzOiAxMAogIGJhdGNoX3NpemU6IDMyCiAgbHI6IDIuMGUtNQogIHdlaWdodF9kZWNheTogMC4wMQogIG9wdGltaXplcjogYWRhbXcKICBzY2hlZHVsZXI6IGNvc2luZQogIHdhcm11cF9zdGVwczogNTAwCiAgYW1wOiB0cnVlCiAgc29mdF9sYWJlbDogdHJ1ZQogIHNlZWQ6IDQyCgpvdXRwdXRfZGlyOiBleHBlcmltZW50cy92aXRfYmVydF9hdHRuCg==", "requirements.txt": "dG9yY2gKdG9yY2h2aXNpb24KdHJhbnNmb3JtZXJzCnNlbnRlbmNlcGllY2UKdGltbQpweXlhbWwKbnVtcHkKcGFuZGFzCnBpbGxvdwp0cWRtCg=="}''')
for path, b64 in FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    with open(path, 'wb') as fh:
        fh.write(base64.b64decode(b64))
open('src/__init__.py', 'a').close()
print('wrote', len(FILES), 'files')
!pip install -q -r requirements.txt

In [ ]:
# 2. Data: mount Drive, copy data.zip (prepared by data_download_VQA.ipynb), unzip.
#    Produces data/train.json, data/valid.json, data/train/, data/valid/
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/data.zip" .
!unzip -q -o data.zip
!ls data

In [ ]:
# 3. Train (safety net first). Swap CONFIG to configs/vit_bert_attn.yaml for the improved run.
CONFIG = 'configs/r50_bert_attn.yaml'
!python -m src.train --config $CONFIG

In [ ]:
# 4. Write submission.npy (answer strings) + model.pt from the best checkpoint
CKPT = 'experiments/r50_bert_attn/best.pt'
!python -m src.predict --config $CONFIG --ckpt $CKPT --out submission/submission.npy

In [ ]:
# 5. Verify the submission format before packaging
import numpy as np
a = np.load('submission/submission.npy')
print('shape', a.shape, 'dtype', a.dtype)
assert a.shape == (4969,), 'test set has 4969 samples'
assert a.dtype.kind in ('U', 'S', 'O'), 'answers must be strings'
print('submission.npy OK')

In [ ]:
# 6. Zip the three required artifacts (<=4.5GB) and submit submission.zip to Omnicampus
import os
from zipfile import ZipFile
NOTEBOOK = '/content/drive/MyDrive/Colab Notebooks/submission.ipynb'  # adjust to this notebook's path
with ZipFile('submission.zip', 'w') as zf:
    zf.write('submission/submission.npy', arcname='submission.npy')
    zf.write('submission/model.pt', arcname='model.pt')
    zf.write(NOTEBOOK, arcname='submission.ipynb')
size_gb = os.path.getsize('submission.zip') / 1e9
print(f'submission.zip = {size_gb:.2f} GB')
assert size_gb <= 4.5, 'zip exceeds 4.5GB limit'